# ENTRENAMIENTO V13: red con decodificador lineal fijo para Curvas J (N bases rampa)

**Cambios V12 → V13 (motivados por el análisis de las salidas de V12):**

*Dataset (requieren reentrenar):*
1. **Etiqueta = óptimo acotado en las cuatro familias (Célula 4):** en V12 las familias `pura` y `combinada` usaban como etiqueta el c generador (sin ruido) mientras la red veía la curva con ruido (|Δc| de hasta 0.028). Ahora la etiqueta es siempre `lsq_linear` sobre la misma curva que ve la red (`ETIQUETA_LSQ_EN_TODAS`). El c generador se conserva en `C_generador` solo como diagnóstico.
2. **Convexificación de las familias de envolvente (Célula 4):** con c ≥ 0 toda combinación de rampas es convexa y creciente. La cota inferior `min(y1, y2)` tiene un quiebre cóncavo en λ ≈ 0.996 y en V12 solo ~50 % de `env_suave` y ~36 % de `env_transicion` eran convexas (≈25 % de `env_transicion` ni siquiera monótonas). Ahora cada curva limpia se reemplaza por su minorante convexa monótona; si eso la deforma más de `TOL_DESVIO_CONVEXO` se descarta y se genera otra (`CONVEXIFICAR_ENVOLVENTE`).

*Consistencia y reporte (no cambian el experimento):*
3. **Célula 2:** el alcance de la caja se reporta para la base más grande y para todas las bases juntas (en V12 se informaba 182.7 kPa como "alcance máximo", que era solo la base más grande).
4. **Célula 1:** el CSV guarda R² relajado (proyección ortogonal sobre span(A)) y R² real (caja), más NRMSE y error máximo.
5. **Célula 3:** funciones únicas de métricas (`metricas_curva`, `metricas_lote`) y de óptimo acotado (`lsq_acotado`) que usan todas las células. NRMSE = RMSE / rango de la propia curva.
6. **Célula 6:** la arquitectura, las activaciones y los pesos de la pérdida se imprimen en texto plano.
7. **Célula 8:** `EarlyStopping` con `min_delta`; historial guardado en CSV; se informa la época de convergencia práctica.
8. **Célula 8b:** proyección explícita sobre la caja con conteo de predicciones crudas fuera de [0, MAX_COEF]; óptimo de cada curva de test; cronometraje justo red (lote) vs. `lsq_linear`.
9. **Células 10 y 11:** el óptimo de referencia pasa a ser `lsq_linear` (el mismo que usan las etiquetas y el test; se retira `trust-constr`, que resolvía el mismo problema). Sobre esa solución se verifican las condiciones KKT y se calculan a posteriori los multiplicadores de las cotas activas, cruzados con el `active_mask` de `lsq_linear`. Se agregan error máximo y exceso de RMSE de la red frente al óptimo.
10. **Célula 12:** métricas por familia contra el óptimo de la misma curva (brecha atribuible a la red), NRMSE por curva (media, mediana, p95), peores casos, R² de coeficientes restringido a las familias de envolvente, residuos relativos con percentiles en escala log (sin ajuste normal) y figura de peores casos.
11. **Célula 12b:** peor R² del óptimo sobre la misma curva, exceso de NRMSE y fracción de curvas OOD que son monótonas y convexas.
12. **Célula 13:** escala vertical por panel y óptimo superpuesto.
13. **Célula 14:** `APLICAR_PISO_ENVOLVENTE = False` por defecto; si se aplica, las métricas se informan también contra la curva cruda. "OLS" pasa a llamarse "óptimo acotado (lsq_linear)".
14. **Célula 14b (nueva):** prueba de reproducción de las bases (4 bases × 4 amplitudes, sin piso de envolvente).

*Fuera del alcance de V13 (pendiente):* redefinición de la envolvente (Tablas 3 y 6 y conversión σθθ → Pᵢ) y reubicación de los nudos de las rampas.

## Célula G — Configuración global (ejecutar SIEMPRE primero)

In [ ]:
# ==============================================================================
# CÉLULA G: CONFIGURACIÓN GLOBAL ÚNICA (constantes, archivos y funciones frontera)
# ==============================================================================
"""
Única fuente de verdad del notebook. Ninguna otra célula redefine estas
constantes ni las funciones frontera. EJECUTAR SIEMPRE PRIMERO (también tras
reiniciar el kernel).
"""
import numpy as np

VERSION = "v13"
SEED = 42

# --- Coeficientes: cota superior ÚNICA ---------------------------------------
COEF_MIN = 0.0
MAX_COEF = 6.0

# --- Dominio y malla unificada ------------------------------------------------
X_MIN_DOMINIO = 0.84
X_MAX_DOMINIO = 1.20
N_PUNTOS_MALLA = 999
X_MESH_999 = np.linspace(X_MIN_DOMINIO, X_MAX_DOMINIO, N_PUNTOS_MALLA)

# --- Umbrales de calidad de las bases -----------------------------------------
UMBRAL_COND = 25.0
UMBRAL_R2 = 0.95

# --- V13: decisiones del dataset ------------------------------------------------
# Etiqueta = óptimo acotado (lsq_linear) sobre la MISMA curva que ve la red, en las 4 familias.
ETIQUETA_LSQ_EN_TODAS = True
# Las rampas con c >= 0 solo generan curvas convexas y crecientes: las curvas de envolvente
# se reemplazan por su minorante convexa monótona antes de agregar ruido.
CONVEXIFICAR_ENVOLVENTE = True
TOL_DESVIO_CONVEXO = 0.15       # desvío máx. |convexa - original| / y_max admitido; si no, se descarta
TOL_FORMA_REL = 0.01            # tolerancia relativa para diagnosticar monotonía / convexidad

# --- Archivos (los usan TODAS las células, incluida la 14) ---------------------
ARCHIVO_BASES = "funciones_base_corregido_v10_intervalo.xlsx"   # bases sin cambios desde V10
ARCHIVO_MODELO = f"modelo_rn_curva_j_pinn_{VERSION}.keras"
ARCHIVO_SCALERS = f"scalers_curva_j_pinn_{VERSION}.pkl"

# --- Funciones frontera biofísicas (Tablas 3 y 6) -----------------------------
# PENDIENTE (fuera de V13): estas fronteras provienen de las Tablas 3 y 6 del documento
# Comparacion_ex_vivo_in_vivo, cuya calibración contra Masson et al. (2011) está en revisión.
H_RI_0_T3 = 0.1703   # Tabla 3 (Jóvenes normotensos)
PA_T3 = 5.63         # kPa
H_RI_0_T6 = 0.2197   # Tabla 6 (Hipertensos tratados)
PA_T6 = 5.98         # kPa


def h_ri_cinematica(x, h_ri_0):
    """h/r_i(x) para la condición de incompresibilidad J = 1."""
    return np.sqrt(1.0 + ((1.0 + h_ri_0) ** 2 - 1.0) / (np.maximum(x, 1e-4) ** 2)) - 1.0


def sigma_bar_tabla3(x):
    return 39.87012495 * (np.exp(7.91934038 * (x - 1.0)) - 1.0)


def sigma_bar_tabla6(x):
    return 73.37535695 * (np.exp(10.40342667 * (x - 1.0)) - 1.0)


def y1_tabla3_exacta(x):
    """Frontera Tabla 3 (Normotensos). OJO: NO es siempre la inferior (cruza a Tabla 6 en x≈0.996)."""
    return sigma_bar_tabla3(x) * np.log(1.0 + h_ri_cinematica(x, H_RI_0_T3)) + PA_T3


def y2_tabla6_exacta(x):
    """Frontera Tabla 6 (Hipertensos)."""
    return sigma_bar_tabla6(x) * np.log(1.0 + h_ri_cinematica(x, H_RI_0_T6)) + PA_T6


# --- Las fronteras se recortan a presión >= PRESION_MIN_KPA (desde V12) -------------
PRESION_MIN_KPA = 0.0


def y1_recortada(x):
    """Frontera Tabla 3 recortada a presión >= PRESION_MIN_KPA."""
    return np.maximum(y1_tabla3_exacta(x), PRESION_MIN_KPA)


def y2_recortada(x):
    """Frontera Tabla 6 recortada a presión >= PRESION_MIN_KPA."""
    return np.maximum(y2_tabla6_exacta(x), PRESION_MIN_KPA)


def envolvente_inferior(x):
    """Mínimo de las dos fronteras. NO es convexa: tiene un quiebre cóncavo donde se cruzan (x≈0.996)."""
    return np.minimum(y1_recortada(x), y2_recortada(x))


def envolvente_superior(x):
    return np.maximum(y1_recortada(x), y2_recortada(x))


# Alias de compatibilidad
y1_limite = envolvente_inferior
y2_limite = envolvente_superior


# --- V13: forma de las curvas (convexidad y monotonía) ---------------------------------
def minorante_convexa_monotona(x, y):
    """Mayor minorante convexa (envolvente convexa inferior) de y, forzada a ser no decreciente.

    max(h(x0), h) es convexa si h lo es, y además no decreciente. Resultado <= y en todo punto.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    idx = []
    for i in range(len(x)):
        while len(idx) >= 2:
            i1, i2 = idx[-2], idx[-1]
            if (y[i2] - y[i1]) * (x[i] - x[i1]) >= (y[i] - y[i1]) * (x[i2] - x[i1]):
                idx.pop()
            else:
                break
        idx.append(i)
    h = np.interp(x, x[idx], y[idx])
    return np.maximum(h, h[0])


def diagnostico_forma(y, x=X_MESH_999, tol_rel=TOL_FORMA_REL):
    """(monótona, convexa) de una curva limpia, con tolerancia relativa a su pendiente máxima."""
    p = np.diff(y) / np.diff(x)
    escala = max(float(np.abs(p).max()), 1e-9)
    return bool(p.min() >= -tol_rel * escala), bool(np.diff(p).min() >= -tol_rel * escala)


print("✅ Configuración global cargada.")
print(f"   • Versión {VERSION} | presión mínima de la envolvente = {PRESION_MIN_KPA} kPa")
print(f"   • MAX_COEF = {MAX_COEF} | Malla: {N_PUNTOS_MALLA} pts en [{X_MIN_DOMINIO}, {X_MAX_DOMINIO}]")
print(f"   • Etiqueta lsq en todas las familias: {ETIQUETA_LSQ_EN_TODAS} | "
      f"Convexificar envolvente: {CONVEXIFICAR_ENVOLVENTE} (tol. desvío {TOL_DESVIO_CONVEXO:.0%})")
print(f"   • Bases: {ARCHIVO_BASES}")
print(f"   • Modelo: {ARCHIVO_MODELO} | Scalers: {ARCHIVO_SCALERS}")

## Célula 0 — Búsqueda evolutiva de bases (opcional, desactivada por defecto)

In [ ]:
# ==============================================================================
# CÉLULA 0 (OPCIONAL): OPTIMIZACIÓN GLOBAL DE LAS BASES CON ALGORITMO EVOLUTIVO
# ==============================================================================
"""
Busca nudos/pendientes/escala de 4 rampas con R2_real alto en todo el dominio
y cond(A) < UMBRAL_COND. NO escribe el Excel: deja el resultado en la variable
PARAMS_BASES_DE. La Célula 2 es la única que construye y exporta las bases
(usa este resultado solo si USAR_RESULTADO_EVOLUTIVO = True allí).
"""
EJECUTAR_EVOLUTIVO = False   # <- poner True para correr la búsqueda (tarda)

if EJECUTAR_EVOLUTIVO:
    import numpy as np
    from scipy.linalg import svd
    from scipy.optimize import lsq_linear, differential_evolution

    x = X_MESH_999
    y_inf = envolvente_inferior(x)
    y_sup = envolvente_superior(x)
    pesos = np.linspace(0.0, 1.0, 11)

    def objetivo(params):
        # params: [k1, k2, k3, m1, m2, m3, m4, escala]
        k1, k2, k3 = params[0], params[1], params[2]
        if not (X_MIN_DOMINIO < k1 < k2 < k3 < X_MAX_DOMINIO):
            return 1e6
        m = params[3:7]
        escala = params[7]

        g1 = m[0] * (x - X_MIN_DOMINIO)
        g2 = m[1] * np.maximum(0.0, x - k1)
        g3 = m[2] * np.maximum(0.0, x - k2)
        g4 = m[3] * np.maximum(0.0, x - k3)
        A_raw = np.column_stack([g1, g2, g3, g4])

        try:
            s = svd(A_raw, compute_uv=False)
            cond = s[0] / s[-1]
        except Exception:
            return 1e6
        if cond >= UMBRAL_COND:
            return cond * 10.0

        A_test = A_raw * escala
        r2_vals = []
        for w in pesos:
            b_test = (1.0 - w) * y_inf + w * y_sup
            res = lsq_linear(A_test, b_test, bounds=(COEF_MIN, MAX_COEF))
            if not res.success:
                return 1e6
            r2 = 1 - np.sum((b_test - A_test @ res.x) ** 2) / np.sum((b_test - b_test.mean()) ** 2)
            r2_vals.append(r2)
        return -min(r2_vals) + (cond / 100.0)

    bounds = [
        (0.85, 0.95), (0.96, 1.05), (1.06, 1.18),          # k1, k2, k3
        (0.5, 2.0), (1.0, 3.0), (2.0, 4.5), (3.0, 6.0),    # m1..m4
        (20.0, 100.0),                                     # escala
    ]

    print("🧬 Ejecutando algoritmo evolutivo (puede tardar)...")
    resultado = differential_evolution(objetivo, bounds, strategy='best1bin',
                                       maxiter=50, popsize=15, seed=SEED, disp=True)
    p = resultado.x
    PARAMS_BASES_DE = dict(knots=list(p[0:3]), pendientes=list(p[3:7]), escala=float(p[7]))

    print("\n" + "=" * 50)
    print("📊 RESULTADO DE LA OPTIMIZACIÓN GLOBAL (guardado en PARAMS_BASES_DE)")
    print("=" * 50)
    for k, v in PARAMS_BASES_DE.items():
        print(f" • {k}: {np.round(v, 4)}")
    print(" ➜ Para usarlo: poner USAR_RESULTADO_EVOLUTIVO = True en la Célula 2.")
else:
    print("ℹ️ Evolutivo omitido (EJECUTAR_EVOLUTIVO = False). La Célula 2 usa los parámetros por defecto.")

## Célula 2 — Construcción y exportación de las bases
> Va **antes** de la Célula 1 porque la 1 verifica el Excel que la 2 escribe.

In [ ]:
# ==============================================================================
# CÉLULA 2: CONSTRUCCIÓN Y EXPORTACIÓN DE LAS BASES (coeficientes en [0, MAX_COEF])
# ==============================================================================
"""
ÚNICA célula que escribe ARCHIVO_BASES. Bases rampa (tipo ReLU):
    A_1(x) = m1·(x − X_MIN)·escala,   A_j(x) = m_j·max(0, x − k_{j−1})·escala  (j = 2..4)
Con c_i >= 0 toda combinación es no negativa, no decreciente y convexa a trozos.
El escalado uniforme no cambia cond(A).
"""
import numpy as np
import pandas as pd
from scipy.linalg import svd

USAR_RESULTADO_EVOLUTIVO = False   # True → usa PARAMS_BASES_DE de la Célula 0

PARAMS_BASES_DEFECTO = dict(
    knots=[0.95, 1.05, 1.1679],
    pendientes=[0.552, 1.2183, 2.0, 6.0],
    escala=100.0,
)

if USAR_RESULTADO_EVOLUTIVO:
    if 'PARAMS_BASES_DE' not in globals():
        raise RuntimeError("PARAMS_BASES_DE no existe: ejecutá la Célula 0 con EJECUTAR_EVOLUTIVO = True.")
    PARAMS_BASES = PARAMS_BASES_DE
else:
    PARAMS_BASES = PARAMS_BASES_DEFECTO


def construir_matriz_bases_escalada(x, knots, pendientes, escala):
    k1, k2, k3 = knots
    m1, m2, m3, m4 = pendientes
    g1 = m1 * (x - X_MIN_DOMINIO)
    g2 = m2 * np.maximum(0.0, x - k1)
    g3 = m3 * np.maximum(0.0, x - k2)
    g4 = m4 * np.maximum(0.0, x - k3)
    return np.column_stack([g1, g2, g3, g4]) * escala


A_opt = construir_matriz_bases_escalada(X_MESH_999, PARAMS_BASES["knots"],
                                        PARAMS_BASES["pendientes"], PARAMS_BASES["escala"])
s_opt = svd(A_opt, compute_uv=False)
cond_opt = s_opt[0] / s_opt[-1]
if cond_opt >= UMBRAL_COND:
    raise ValueError(f"cond(A) = {cond_opt:.2f} >= {UMBRAL_COND}: bases rechazadas.")

y_sup_max = float(envolvente_superior(X_MESH_999).max())
# Alcance de la caja [0, MAX_COEF] (las rampas son crecientes: el máximo está en x = X_MAX_DOMINIO)
alcance_base_mayor = float(A_opt.max(axis=0).max() * MAX_COEF)   # una sola base (la mayor) en MAX_COEF
alcance_todas = float(A_opt.sum(axis=1).max() * MAX_COEF)        # todas las bases en MAX_COEF

print("\n" + "=" * 70)
print(f"📊 REPORTE DE BASES ESCALADAS PARA CAJA [0, {MAX_COEF:g}]")
print("=" * 70)
print(f" • Nudos [k1, k2, k3] : {PARAMS_BASES['knots']}")
print(f" • Pendientes base    : {PARAMS_BASES['pendientes']}")
print(f" • Factor de escala   : {PARAMS_BASES['escala']}")
print(f" • cond(A) resultante : {cond_opt:.4f}")
print(f" • Máx. por base (x={X_MAX_DOMINIO}) con c=1: {np.round(A_opt[-1], 2)}")
print(f" • y_sup máximo de la envolvente = {y_sup_max:.1f} kPa")
print(f" • Alcance con c <= {MAX_COEF:g}: base mayor sola = {alcance_base_mayor:.1f} kPa | "
      f"todas las bases = {alcance_todas:.1f} kPa")
print("=" * 70)

pd.DataFrame(np.column_stack([X_MESH_999, A_opt])).to_excel(ARCHIVO_BASES, header=False, index=False)
print(f"💾 Archivo guardado con éxito en '{ARCHIVO_BASES}'.")

## Célula 1 — Verificación de R² y cond(A) de las bases

In [ ]:
# ==============================================================================
# CÉLULA 1: VERIFICACIÓN DE LAS BASES — PROYECCIÓN SIN Y CON CAJA SOBRE LAS FRONTERAS
# ==============================================================================
"""
Lee las bases desde el Excel y evalúa, para las mezclas b_w = (1−w)·y_inf + w·y_sup:
  • R² relajado: proyección ortogonal sobre span(A) (mínimos cuadrados sin cotas, vía SVD).
  • R² real    : mínimos cuadrados con la caja [0, MAX_COEF] (lsq_linear).
Son 11 curvas de una sola familia: miden el piso de las bases sobre las fronteras,
NO sobre el dataset (eso se reporta por familia en la Célula 12).
"""
import os
import numpy as np
import pandas as pd
from scipy.linalg import svd
from scipy.optimize import lsq_linear

if not os.path.exists(ARCHIVO_BASES):
    raise FileNotFoundError(f"No se encuentra '{ARCHIVO_BASES}'. Ejecutá antes la Célula 2.")

arr = pd.read_excel(ARCHIVO_BASES, header=None).to_numpy(float)
x, A = arr[:, 0], arr[:, 1:]
n_bases = A.shape[1]
print(f"📊 Malla leída: {len(x)} puntos en [{x[0]:.2f}, {x[-1]:.2f}] con {n_bases} bases activas.")

U, s, _ = svd(A, full_matrices=False)
cond_A = s[0] / s[-1]
estado = "OK" if cond_A < UMBRAL_COND else "FUERA DE UMBRAL"
print(f"   • cond(A) = {cond_A:.2f} (umbral {UMBRAL_COND}) → {estado}\n")

Ue = U[:, s > 1e-10 * s[0]]


def _r2(b, b_hat):
    return 1 - np.sum((b - b_hat) ** 2) / np.sum((b - b.mean()) ** 2)


y_inf = envolvente_inferior(x)
y_sup = envolvente_superior(x)
pesos = np.linspace(0.0, 1.0, 11)
filas = []

print(f"{'w':>5} {'R2_relajado':>12} {'R2_real':>10} {'NRMSE_real':>11} {'MaxAE_real':>11} {'cotas activas':>14}")
print("-" * 68)
for w in pesos:
    b_test = (1.0 - w) * y_inf + w * y_sup
    b_rel = Ue @ (Ue.T @ b_test)
    sol = lsq_linear(A, b_test, bounds=(COEF_MIN, MAX_COEF))
    b_box = A @ sol.x
    rango = b_test.max() - b_test.min()
    nrmse = np.sqrt(np.mean((b_test - b_box) ** 2)) / rango * 100.0
    maxae = np.max(np.abs(b_test - b_box))
    n_act = int(np.sum(sol.active_mask != 0))
    filas.append(dict(peso_envolvente=w, R2_relajado=_r2(b_test, b_rel), R2_real=_r2(b_test, b_box),
                      NRMSE_real_pct=nrmse, MaxAE_real_kPa=maxae, cotas_activas=n_act))
    print(f"{w:5.2f} {filas[-1]['R2_relajado']:12.4f} {filas[-1]['R2_real']:10.4f} "
          f"{nrmse:10.3f}% {maxae:10.3f} {n_act:>14d}")

df_verif = pd.DataFrame(filas)
r2_min = df_verif["R2_real"].min()
print(f"\n📈 Promedio de R2_real: {df_verif['R2_real'].mean():.4f} | mínimo: {r2_min:.4f} "
      f"(umbral {UMBRAL_R2}) → {'OK' if r2_min >= UMBRAL_R2 else 'BAJO UMBRAL'}")
print("   (R2_relajado = R2_real indica que la caja no está activa en el óptimo de esas curvas)")

ARCHIVO_VERIF = f"verificacion_proyeccion_bases_{VERSION}.csv"
df_verif.to_csv(ARCHIVO_VERIF, index=False)
print(f"💾 Archivo '{ARCHIVO_VERIF}' guardado.")

## Célula 3 — Librerías, entorno TensorFlow y transformación logarítmica

In [ ]:
# ==============================================================================
# CÉLULA 3: LIBRERÍAS, ENTORNO TENSORFLOW, TRANSFORMACIÓN LOGARÍTMICA Y MÉTRICAS ÚNICAS
# ==============================================================================
import os
# Variables de entorno ANTES de importar TensorFlow (si no, no tienen efecto)
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pickle
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import lsq_linear
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import tensorflow as tf

warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"Aviso GPU: {e}")
else:
    print("ℹ️ Sin GPU visible para TensorFlow: el entrenamiento corre en CPU (no cambia los resultados).")

np.random.seed(SEED)
tf.random.set_seed(SEED)
RNG = np.random.default_rng(SEED)

# ------------------------------------------------------------------------------
# TRANSFORMACIÓN LOGARÍTMICA DE LAS ENTRADAS: X = log(y + OFFSET_LOG)
# ------------------------------------------------------------------------------
# OFFSET_LOG = |mínimo de la envolvente| + 5 kPa. Con la envolvente recortada a >= 0 vale 5:
# ya no evita logaritmos de negativos, sino que suaviza la compresión del log en el pie de la curva.
_y_min_possible = float(min(envolvente_inferior(X_MESH_999).min(),
                            envolvente_superior(X_MESH_999).min()))
OFFSET_LOG = float(np.abs(_y_min_possible) + 5.0)
EPSILON_LOG = 1e-6


def safe_log_transform(y_data, offset=None):
    """Transformación logarítmica blindada contra NaNs e Infs."""
    if offset is None:
        offset = OFFSET_LOG
    y_safe = np.maximum(np.asarray(y_data) + offset, EPSILON_LOG)
    return np.nan_to_num(np.log(y_safe), nan=0.0, posinf=15.0, neginf=-15.0)


# ------------------------------------------------------------------------------
# ÓPTIMO ACOTADO Y MÉTRICAS (ÚNICA DEFINICIÓN PARA TODO EL NOTEBOOK)
# ------------------------------------------------------------------------------
def lsq_acotado(A, y):
    """Óptimo de mínimos cuadrados con la caja [COEF_MIN, MAX_COEF] (techo alcanzable con las bases)."""
    return lsq_linear(A, y, bounds=(COEF_MIN, MAX_COEF)).x


def metricas_curva(y_real, y_aprox):
    """Métricas de UNA curva. NRMSE y MaxAE_rel en % del rango de la propia curva."""
    y_real = np.asarray(y_real, dtype=float)
    err = y_real - np.asarray(y_aprox, dtype=float)
    rango = float(y_real.max() - y_real.min())
    ss_tot = float(np.sum((y_real - y_real.mean()) ** 2))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    maxae = float(np.max(np.abs(err)))
    return {"R2": 1.0 - float(np.sum(err ** 2)) / ss_tot if ss_tot > 0 else np.nan,
            "RMSE": rmse, "MAE": float(np.mean(np.abs(err))), "MaxAE": maxae,
            "NRMSE": rmse / rango * 100.0 if rango > 0 else np.nan,
            "MaxAE_rel": maxae / rango * 100.0 if rango > 0 else np.nan}


def metricas_lote(Y_real, Y_aprox):
    """Mismas métricas que metricas_curva, vectorizadas por filas (una curva por fila)."""
    Y_real = np.asarray(Y_real, dtype=float)
    err = Y_real - np.asarray(Y_aprox, dtype=float)
    rango = Y_real.max(axis=1) - Y_real.min(axis=1)
    ss_tot = np.sum((Y_real - Y_real.mean(axis=1, keepdims=True)) ** 2, axis=1)
    rmse = np.sqrt(np.mean(err ** 2, axis=1))
    maxae = np.max(np.abs(err), axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return {"R2": np.where(ss_tot > 0, 1.0 - np.sum(err ** 2, axis=1) / ss_tot, np.nan),
                "RMSE": rmse, "MAE": np.mean(np.abs(err), axis=1), "MaxAE": maxae,
                "NRMSE": np.where(rango > 0, rmse / rango * 100.0, np.nan),
                "MaxAE_rel": np.where(rango > 0, maxae / rango * 100.0, np.nan)}


print("✅ Entorno inicializado.")
print(f" Malla unificada: {N_PUNTOS_MALLA} puntos en x ∈ [{X_MIN_DOMINIO}, {X_MAX_DOMINIO}]")
print(f"🛡️ OFFSET_LOG = {OFFSET_LOG:.4f} (mínimo de la envolvente: {_y_min_possible:.4f} kPa)")
print(" Métricas: NRMSE = RMSE / rango de la propia curva (%); MaxAE_rel = error máximo / rango (%).")

## Célula 4 — Dataset sintético V13 (4 familias, etiqueta lsq en todas, envolvente convexificada)

In [ ]:
# ==============================================================================
# CÉLULA 4: DATASET SINTÉTICO V13 (ENTRADA Y ETIQUETA SIEMPRE CONSISTENTES)
# ==============================================================================
"""
Regla de oro: la etiqueta c es SIEMPRE el óptimo acotado (lsq_linear) de la MISMA curva
que ve la red, ruido incluido (ETIQUETA_LSQ_EN_TODAS = True). No hay clip de las curvas
contra la envolvente; el único recorte es y >= 0 y se aplica ANTES de calcular la etiqueta.

Familias (vector 'familia'):
  0) pura          : una sola base activa, c_gen ~ U(C_MIN_PURA, MAX_COEF) + ruido.
  1) combinada     : 2..N bases activas, c_gen ~ U(0, MAX_COEF), reescaladas si la curva supera
                     Y_TECHO_KPA, + ruido.
  2) env_suave     : dentro de la envolvente, alpha(x) casi constante + cosenos pequeños.
  3) env_transicion: dentro de la envolvente, alpha(x) con transición sigmoide (saltos de hasta ~1).
Las familias 0 y 1 están mayormente FUERA de la envolvente por construcción (una rampa sola queda
bajo la cota inferior): cumplen el rol de cubrir el espacio de coeficientes. Las métricas del
dominio objetivo se reportan por familia (Célula 12).

V13 — Convexificación (CONVEXIFICAR_ENVOLVENTE): con c >= 0 las rampas solo generan curvas
convexas y crecientes. La curva limpia de las familias 2 y 3 se reemplaza por su minorante convexa
monótona; si el desvío relativo supera TOL_DESVIO_CONVEXO se descarta y se sortea otra.
"""
import os
import numpy as np
import pandas as pd
from scipy.optimize import lsq_linear

# --- Parámetros del dataset ---------------------------------------------------
N_SAMPLES = 30000
FRAC_PURAS = 0.25
FRAC_COMBINADAS = 0.25
FRAC_ENV_SUAVE = 0.25
FRAC_ENV_TRANS = 1.0 - FRAC_PURAS - FRAC_COMBINADAS - FRAC_ENV_SUAVE
C_MIN_PURA = 0.05          # evita curvas puras idénticamente nulas
Y_TECHO_KPA = 200.0        # techo de amplitud de las curvas combinadas
RUIDO_REL_PURAS = 0.005
RUIDO_REL_ENVOLVENTE = 0.01
ALPHA_MIN, ALPHA_MAX = -0.05, 1.05   # un poco más allá de las fronteras (robustez en los bordes)
P_TRANS_EXTREMA = 0.5      # prob. de que la transición vaya de ~0 a ~1 (o al revés)
MAX_INTENTOS_CURVA = 50    # sorteos máximos por curva de envolvente antes de abortar
SEED_DATASET = 2026
FAMILIAS = {0: "pura", 1: "combinada", 2: "env_suave", 3: "env_transicion"}

rng_ds = np.random.default_rng(SEED_DATASET)
print(f"⚙️ Generando dataset {VERSION.upper()}: {N_SAMPLES} curvas "
      f"({FRAC_PURAS:.0%} puras / {FRAC_COMBINADAS:.0%} combinadas / "
      f"{FRAC_ENV_SUAVE:.0%} env_suave / {FRAC_ENV_TRANS:.0%} env_transición)...")

# --- 1. Carga de la matriz de bases (dinámica en N) ----------------------------
if not os.path.exists(ARCHIVO_BASES):
    raise FileNotFoundError(f"No se encontró '{ARCHIVO_BASES}'. Ejecutá la Célula 2.")

df_base = pd.read_excel(ARCHIVO_BASES, header=None)
x_base_file = df_base.iloc[:, 0].values.astype(np.float64)
A_base_full = df_base.iloc[:, 1:].values.astype(np.float64)
n_bases = A_base_full.shape[1]

if len(x_base_file) != N_PUNTOS_MALLA or not np.allclose(x_base_file, X_MESH_999):
    A_base = np.column_stack([np.interp(X_MESH_999, x_base_file, A_base_full[:, j])
                              for j in range(n_bases)])
else:
    A_base = A_base_full.copy()
print(f"📂 Bases cargadas: {n_bases} bases en {N_PUNTOS_MALLA} puntos.")

cota_inferior = envolvente_inferior(X_MESH_999)
cota_superior = envolvente_superior(X_MESH_999)
t = (X_MESH_999 - X_MIN_DOMINIO) / (X_MAX_DOMINIO - X_MIN_DOMINIO)

n_puras = int(round(FRAC_PURAS * N_SAMPLES))
n_comb = int(round(FRAC_COMBINADAS * N_SAMPLES))
n_suave = int(round(FRAC_ENV_SUAVE * N_SAMPLES))
n_trans = N_SAMPLES - n_puras - n_comb - n_suave

Y_raw = np.zeros((N_SAMPLES, N_PUNTOS_MALLA), dtype=np.float64)
C_targets = np.zeros((N_SAMPLES, n_bases), dtype=np.float64)
C_generador = np.full((N_SAMPLES, n_bases), np.nan)   # solo familias 0 y 1 (diagnóstico)
familia = np.zeros(N_SAMPLES, dtype=np.int8)
pos = 0
DIAG_CONVEXO = {}   # estadísticas de la convexificación por familia


def _guardar(Y, C, fam_id, C_gen=None):
    global pos
    n = len(Y)
    Y_raw[pos:pos + n], C_targets[pos:pos + n], familia[pos:pos + n] = Y, C, fam_id
    if C_gen is not None:
        C_generador[pos:pos + n] = C_gen
    pos += n


def _etiquetar(Y, C_gen):
    """Etiqueta coherente con la curva que ve la red."""
    if ETIQUETA_LSQ_EN_TODAS:
        return np.array([lsq_acotado(A_base, y) for y in Y])
    return C_gen.copy()


# --- A) PURAS (balanceadas entre bases, rango completo de c) -------------------
base_idx = rng_ds.permutation(np.arange(n_puras) % n_bases)
C_p = np.zeros((n_puras, n_bases))
C_p[np.arange(n_puras), base_idx] = rng_ds.uniform(C_MIN_PURA, MAX_COEF, size=n_puras)
Y_p = C_p @ A_base.T
Y_p = Y_p + rng_ds.normal(0.0, RUIDO_REL_PURAS, size=Y_p.shape) * np.abs(Y_p)
Y_p = np.maximum(Y_p, PRESION_MIN_KPA)
_guardar(Y_p, _etiquetar(Y_p, C_p), 0, C_p)

# --- B) COMBINADAS (soporte disperso aleatorio) ----------------------------------
C_c = np.zeros((n_comb, n_bases))
for i in range(n_comb):
    k = rng_ds.integers(2, n_bases + 1)                      # nº de bases activas (2..N)
    act = rng_ds.choice(n_bases, size=k, replace=False)
    C_c[i, act] = rng_ds.uniform(0.0, MAX_COEF, size=k)
Y_c = C_c @ A_base.T
y_max_c = Y_c.max(axis=1, keepdims=True)
factor = np.minimum(1.0, Y_TECHO_KPA / np.maximum(y_max_c, 1e-9))   # reescala c (sigue <= MAX_COEF)
C_c = C_c * factor
Y_c = C_c @ A_base.T
Y_c = Y_c + rng_ds.normal(0.0, RUIDO_REL_PURAS, size=Y_c.shape) * np.abs(Y_c)
Y_c = np.maximum(Y_c, PRESION_MIN_KPA)
_guardar(Y_c, _etiquetar(Y_c, C_c), 1, C_c)


# --- C) y D) ENVOLVENTE: alpha(x) suave / con transición ------------------------------
def alpha_suave():
    a = rng_ds.uniform(ALPHA_MIN, ALPHA_MAX)
    for kk in (1, 2, 3):
        a = a + rng_ds.normal(0.0, 0.10 / kk) * np.cos(kk * np.pi * t)
    return np.clip(a, ALPHA_MIN, ALPHA_MAX)


def alpha_transicion():
    if rng_ds.random() < P_TRANS_EXTREMA:                      # salto casi completo 0↔1
        a_bajo = rng_ds.uniform(ALPHA_MIN, 0.10)
        a_alto = rng_ds.uniform(0.90, ALPHA_MAX)
        a_izq, a_der = (a_bajo, a_alto) if rng_ds.random() < 0.5 else (a_alto, a_bajo)
    else:
        a_izq, a_der = rng_ds.uniform(ALPHA_MIN, ALPHA_MAX, size=2)
    x0 = rng_ds.uniform(0.88, 1.15)                            # posición de la transición
    w = 10.0 ** rng_ds.uniform(-2.7, -1.1)                     # ancho: 0.002 (casi escalón) a 0.08
    s = 1.0 / (1.0 + np.exp(-np.clip((X_MESH_999 - x0) / w, -60.0, 60.0)))
    a = a_izq + (a_der - a_izq) * s
    for kk in (1, 2):
        a = a + rng_ds.normal(0.0, 0.05 / kk) * np.cos(kk * np.pi * t)
    return np.clip(a, ALPHA_MIN, ALPHA_MAX)


def generar_envolvente(n, fn_alpha, nombre):
    Y = np.zeros((n, N_PUNTOS_MALLA))
    n_desc, n_noconv_orig, desvios = 0, 0, []
    for i in range(n):
        for _ in range(MAX_INTENTOS_CURVA):
            y0 = np.maximum(cota_inferior + fn_alpha() * (cota_superior - cota_inferior), PRESION_MIN_KPA)
            if not CONVEXIFICAR_ENVOLVENTE:
                break
            h = minorante_convexa_monotona(X_MESH_999, y0)
            desvio = float(np.max(np.abs(h - y0)) / max(y0.max(), 1.0))
            if desvio <= TOL_DESVIO_CONVEXO:
                if not all(diagnostico_forma(y0)):
                    n_noconv_orig += 1
                desvios.append(desvio)
                y0 = h
                break
            n_desc += 1
        else:
            raise RuntimeError(f"{nombre}: {MAX_INTENTOS_CURVA} sorteos sin una curva convexificable.")
        y = y0 + rng_ds.normal(0.0, RUIDO_REL_ENVOLVENTE, size=N_PUNTOS_MALLA) * np.abs(y0)
        Y[i] = np.maximum(y, PRESION_MIN_KPA)                           # recorte ANTES de la etiqueta
    if CONVEXIFICAR_ENVOLVENTE:
        DIAG_CONVEXO[nombre] = dict(descartadas=n_desc, corregidas=n_noconv_orig,
                                    desvio_medio=float(np.mean(desvios)), desvio_p95=float(np.percentile(desvios, 95)))
    C = np.array([lsq_acotado(A_base, y) for y in Y])                   # misma curva que ve la red
    return Y, C


_guardar(*generar_envolvente(n_suave, alpha_suave, "env_suave"), 2)
_guardar(*generar_envolvente(n_trans, alpha_transicion, "env_transicion"), 3)
assert pos == N_SAMPLES

# Alias de compatibilidad
X_curvas = Y_raw
Y_coef = C_targets

# --- 2. Transformación logarítmica de las entradas -----------------------------
X_log = safe_log_transform(Y_raw, offset=OFFSET_LOG)

# --- 3. Chequeos de integridad y consistencia entrada↔etiqueta -----------------
assert np.isfinite(X_log).all() and np.isfinite(C_targets).all(), "NaN/Inf en el dataset"
assert C_targets.min() >= COEF_MIN - 1e-9 and C_targets.max() <= MAX_COEF + 1e-9, \
    f"Etiquetas fuera de [0, {MAX_COEF}]"

print(f"\n✅ Dataset {VERSION.upper()} generado:")
print(f"   • X_log: {X_log.shape} | C_targets: {C_targets.shape}")
for f, nombre in FAMILIAS.items():
    m = familia == f
    fuera = np.mean((Y_raw[m] < cota_inferior - 1e-9) | (Y_raw[m] > cota_superior + 1e-9))
    print(f"   • {nombre:<14}: {m.sum():>6} curvas | fracción de puntos fuera de la envolvente: {fuera:6.1%}")

if CONVEXIFICAR_ENVOLVENTE:
    print("\n🔺 Convexificación de las familias de envolvente (curva limpia → minorante convexa monótona):")
    for nombre, d in DIAG_CONVEXO.items():
        print(f"   • {nombre:<14}: corregidas {d['corregidas']:>5} | descartadas {d['descartadas']:>5} | "
              f"desvío relativo medio {d['desvio_medio']:.2%} (p95 {d['desvio_p95']:.2%})")

print("\n📏 Cobertura de coeficientes (percentiles 5/50/95 y % de muestras con c > 0.7·MAX_COEF):")
for j in range(n_bases):
    p5, p50, p95 = np.percentile(C_targets[:, j], [5, 50, 95])
    print(f"   c{j+1}: p5={p5:5.2f}  p50={p50:5.2f}  p95={p95:5.2f}  | c>{0.7*MAX_COEF:.1f}: {np.mean(C_targets[:, j] > 0.7*MAX_COEF):6.1%}")
m_env = familia >= 2
print(f"   Familias de envolvente: c máximo por coeficiente = {np.round(C_targets[m_env].max(axis=0), 3)}")

print("\n🔍 Consistencia entrada↔etiqueta (re-resolviendo lsq_linear sobre 200 curvas por familia; esperado ≈ 0):")
for f, nombre in FAMILIAS.items():
    idx = np.where(familia == f)[0]
    idx = rng_ds.choice(idx, size=min(200, len(idx)), replace=False)
    dif = np.array([np.abs(lsq_acotado(A_base, Y_raw[i]) - C_targets[i]).max() for i in idx])
    print(f"   • {nombre:<14}: |Δc| máx. medio = {dif.mean():.2e} | peor = {dif.max():.2e}")

print("\n🧪 Efecto del ruido en las familias 0 y 1 (etiqueta lsq vs. c generador; era el error de etiqueta de V12):")
for f in (0, 1):
    m = familia == f
    d = np.abs(C_targets[m] - C_generador[m]).max(axis=1)
    print(f"   • {FAMILIAS[f]:<14}: |c_lsq − c_gen| máx. medio = {d.mean():.4f} | peor = {d.max():.4f}")

## Célula 5 — Partición train/val/test y normalización sin fuga

In [ ]:
# ==============================================================================
# CÉLULA 5: PARTICIÓN TRAIN / VAL / TEST Y NORMALIZACIÓN (SIN FUGA DE INFORMACIÓN)
# ==============================================================================
"""
Los escaladores se ajustan SOLO con train. La validación (early stopping y
checkpoint) es distinta del test, que queda intacto para la evaluación final.
La partición es estratificada por familia de curva.
"""
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

idx_all = np.arange(len(X_log))
idx_trval, idx_test = train_test_split(
    idx_all, test_size=0.20, random_state=SEED, stratify=familia)
idx_train, idx_val = train_test_split(
    idx_trval, test_size=0.125, random_state=SEED, stratify=familia[idx_trval])   # 0.125·0.8 = 10 %

sx = StandardScaler().fit(X_log[idx_train])      # entradas
sy = StandardScaler().fit(Y_coef[idx_train])     # salidas (coeficientes)

X_train, X_val, X_test = (sx.transform(X_log[i]) for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = (sy.transform(Y_coef[i]) for i in (idx_train, idx_val, idx_test))
fam_train, fam_val, fam_test = familia[idx_train], familia[idx_val], familia[idx_test]

print("✅ Datos normalizados y divididos (escaladores ajustados solo con train):")
print(f"   • Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"   • X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"   • sy.mean_  : {np.round(sy.mean_, 4)}")
print(f"   • sy.scale_ : {np.round(sy.scale_, 4)}")

## Célula 6 — Pérdida compuesta V13 y arquitectura (LayerNormalization)

In [ ]:
# ==============================================================================
# CÉLULA 6: PÉRDIDA COMPUESTA V13 Y ARQUITECTURA (LAYER NORMALIZATION)
# ==============================================================================
"""
Pérdida = PESO_COEF · MSE(coeficientes estandarizados)
        + PESO_REC  · MSE(reconstrucción A·c relativa al y_max de cada curva)   [unidades físicas]
        + PESO_CROSSTALK · media(c_pred² en canales con etiqueta < 1e-4)          [unidades físicas]
La reconstrucción y el crosstalk se calculan DESPUÉS de desestandarizar y de proyectar c_pred >= 0,
así que el umbral 1e-4 y la no negatividad están en unidades físicas.
La salida es lineal (espacio estandarizado); en inferencia los coeficientes se proyectan sobre la
caja [0, MAX_COEF] y se informa cuántos caían fuera (Célula 8b).
"""
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LayerNormalization
from tensorflow.keras.regularizers import l2

# --- 1. PARÁMETROS (derivados de los datos, sin respaldos por globals()) -----------
L2_REGULARIZATION = 1e-5
PESO_COEF, PESO_REC, PESO_CROSSTALK = 1.0, 5.0, 20.0
UMBRAL_CERO_CROSSTALK = 1e-4
CAPAS_OCULTAS = (256, 128, 64)
LEARNING_RATE_INICIAL = 1e-3

N_BASES = y_train.shape[1]
N_PUNTOS = X_train.shape[1]
assert A_base.shape == (N_PUNTOS, N_BASES), \
    f"A_base {A_base.shape} no coincide con (N_PUNTOS, N_BASES) = ({N_PUNTOS}, {N_BASES})"

# A en forma (N_BASES, N_PUNTOS) para: coef (batch, N_BASES) @ A -> curva (batch, N_PUNTOS)
A_mat_np = A_base.T.astype(np.float32)
mean_arr = np.asarray(sy.mean_, dtype=np.float32)
scale_arr = np.asarray(sy.scale_, dtype=np.float32)


# --- 2. PÉRDIDA (constantes estáticas de TF) --------------------------------------------
def crear_loss_reconstruccion(A_np, mean_np, scale_np,
                              peso_coef=PESO_COEF, peso_rec=PESO_REC, peso_crosstalk=PESO_CROSSTALK):
    A_tf = tf.constant(A_np, dtype=tf.float32)
    mean_tf = tf.constant(mean_np, dtype=tf.float32)
    scale_tf = tf.constant(scale_np, dtype=tf.float32)

    def loss(y_true, y_pred):
        # A. Pérdida paramétrica en espacio estandarizado
        loss_coef = tf.reduce_mean(tf.square(y_true - y_pred))

        # B. Desestandarización a unidades físicas
        coef_true = y_true * scale_tf + mean_tf
        coef_pred = y_pred * scale_tf + mean_tf

        # C. No negatividad en unidades físicas (c_i >= 0)
        coef_pred_pos = tf.maximum(coef_pred, 0.0)
        coef_true_pos = tf.maximum(coef_true, 0.0)

        # D. Reconstrucción física
        b_true = tf.matmul(coef_true_pos, A_tf)
        b_pred = tf.matmul(coef_pred_pos, A_tf)

        # E. Error de reconstrucción relativo a la amplitud de cada curva
        y_max_safe = tf.maximum(tf.reduce_max(b_true, axis=1, keepdims=True), 1.0)
        loss_rec = tf.reduce_mean(tf.square((b_true - b_pred) / (y_max_safe + 1e-6)))

        # F. Crosstalk: canales cuya etiqueta física es ~0
        mascara_ceros = tf.cast(tf.less(coef_true_pos, UMBRAL_CERO_CROSSTALK), tf.float32)
        loss_cross = tf.reduce_mean(tf.square(coef_pred_pos) * mascara_ceros)

        return peso_coef * loss_coef + peso_rec * loss_rec + peso_crosstalk * loss_cross

    return loss


loss_pinn = crear_loss_reconstruccion(A_mat_np, mean_arr, scale_arr)


# --- 3. ARQUITECTURA ----------------------------------------------------------------------
def construir_modelo(input_shape, num_outputs):
    capas = [Input(shape=(input_shape,))]
    for n in CAPAS_OCULTAS:
        capas += [Dense(n, activation='relu', kernel_regularizer=l2(L2_REGULARIZATION)), LayerNormalization()]
    capas.append(Dense(num_outputs, activation='linear'))
    return Sequential(capas)


model = construir_modelo(input_shape=N_PUNTOS, num_outputs=N_BASES)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_INICIAL, clipnorm=1.0)
model.compile(optimizer=optimizer, loss=loss_pinn, metrics=['mae', 'mse'])


# --- 4. RESUMEN EN TEXTO PLANO (copiable al artículo) --------------------------------------
def imprimir_resumen_modelo(m):
    print(f"{'Capa':<24} {'Tipo':<20} {'Activación':<11} {'Salida':<14} {'Parámetros':>11}")
    print("-" * 84)
    for capa in m.layers:
        act = getattr(capa, "activation", None)
        act = act.__name__ if act is not None else "—"
        print(f"{capa.name:<24} {capa.__class__.__name__:<20} {act:<11} "
              f"{str(capa.output.shape):<14} {capa.count_params():>11,d}")
    print("-" * 84)
    print(f"{'Total de parámetros':<71} {m.count_params():>11,d}")


print("✅ Pérdida compuesta y modelo construidos:")
print(f"   • Arquitectura: {N_PUNTOS} → {' → '.join(map(str, CAPAS_OCULTAS))} → {N_BASES} "
      f"(ReLU + LayerNormalization; salida lineal)")
print(f"   • Pesos de la pérdida: coef = {PESO_COEF} | reconstrucción = {PESO_REC} | crosstalk = {PESO_CROSSTALK} "
      f"(umbral de cero {UMBRAL_CERO_CROSSTALK:g})")
print(f"   • L2 = {L2_REGULARIZATION:g} | Adam lr0 = {LEARNING_RATE_INICIAL:g}, clipnorm = 1.0\n")
imprimir_resumen_modelo(model)

## Célula 7 — Diagnóstico de integridad

In [ ]:
# ==============================================================================
# CÉLULA 7: DIAGNÓSTICO DE INTEGRIDAD DE DATOS (EJECUTAR ANTES DE MODEL.FIT)
# ==============================================================================
import numpy as np

print("🔍 VERIFICACIÓN DE INTEGRIDAD DE TENSORES:")
for nombre, arr in [("X_train", X_train), ("y_train", y_train),
                    ("X_val", X_val), ("y_val", y_val),
                    ("X_test", X_test), ("y_test", y_test)]:
    print(f" • {nombre:<8} {str(arr.shape):<14} NaNs: {np.isnan(arr).sum()} | Infs: {np.isinf(arr).sum()}")

print(f" • Scaler X → medias con NaN: {np.isnan(sx.mean_).sum()} | Scaler Y → medias con NaN: {np.isnan(sy.mean_).sum()}")

c_all = sy.inverse_transform(np.vstack([y_train, y_val, y_test]))
print(f" • Etiquetas físicas: min = {c_all.min():.4f} | max = {c_all.max():.4f} (caja [0, {MAX_COEF:g}])")
assert c_all.min() >= -1e-6 and c_all.max() <= MAX_COEF + 1e-6, "Etiquetas fuera de la caja"
print(" ✅ Integridad OK.")

## Célula 8 — Entrenamiento y guardado de modelo + escaladores

In [ ]:
# ==============================================================================
# CÉLULA 8: ENTRENAMIENTO V13 Y GUARDADO DE MODELO + ESCALADORES
# ==============================================================================
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TerminateOnNaN

PATIENCE_EARLY_STOP = 35
PATIENCE_LR = 15
MIN_DELTA_VAL = 1e-5        # mejora mínima de val_loss que cuenta (V12 seguía "mejorando" en el 5.º decimal)
EPOCHS = 1000               # tope; el corte real lo decide EarlyStopping
BATCH_SIZE = 64
TOL_CONVERGENCIA = 0.01     # época de convergencia práctica: val_loss dentro del 1 % del mejor valor

callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor='val_loss', patience=PATIENCE_EARLY_STOP, min_delta=MIN_DELTA_VAL,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_LR, min_delta=MIN_DELTA_VAL,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath=ARCHIVO_MODELO, monitor='val_loss', save_best_only=True, verbose=0),
]

print(f"🚀 Entrenamiento {VERSION.upper()} para N = {N_BASES} bases ({N_PUNTOS} pts) | validación ≠ test...")
t_start = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),      # el test NO participa en el entrenamiento ni en la selección
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

t_elapsed = time.time() - t_start
hist = pd.DataFrame(history.history)
hist.index = np.arange(1, len(hist) + 1)
hist.index.name = "epoca"
val_hist = hist['val_loss'].to_numpy()
best_ep = int(np.argmin(val_hist)) + 1
ep_conv = int(np.argmax(val_hist <= val_hist.min() * (1.0 + TOL_CONVERGENCIA))) + 1
lr_col = next((c for c in ("learning_rate", "lr") if c in hist.columns), None)

print(f"\n✅ Entrenamiento completado en {t_elapsed/60:.2f} min ({len(val_hist)} épocas ejecutadas).")
print(f"   • Mejor época: {best_ep} | Mejor val_loss: {val_hist.min():.6f} | "
      f"loss de entrenamiento en esa época: {hist['loss'].iloc[best_ep - 1]:.6f}")
print(f"   • Convergencia práctica (val_loss a ≤ {TOL_CONVERGENCIA:.0%} del mejor): época {ep_conv}")
print(f"   • MAE de validación en la mejor época (espacio estandarizado): {hist['val_mae'].iloc[best_ep - 1]:.6f}")
if lr_col:
    print(f"   • Learning rate final: {hist[lr_col].iloc[-1]:.2e}")

# --- Guardado coherente con la Célula 14 -------------------------------------------------
model.save(ARCHIVO_MODELO)
with open(ARCHIVO_SCALERS, "wb") as f:
    pickle.dump({
        "scaler_x": sx,
        "scaler_y": sy,
        "offset_log": OFFSET_LOG,
        "max_coef": MAX_COEF,
        "n_bases": N_BASES,
        "x_mesh": X_MESH_999,
        "version": VERSION,
    }, f)
ARCHIVO_HISTORIAL = f"historial_entrenamiento_{VERSION}.csv"
hist.to_csv(ARCHIVO_HISTORIAL)
print(f"💾 Modelo → '{ARCHIVO_MODELO}' | Escaladores + OFFSET_LOG → '{ARCHIVO_SCALERS}' | "
      f"Historial → '{ARCHIVO_HISTORIAL}'")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist.index, hist['loss'], label='Loss Entrenamiento', color='#1d4ed8', linewidth=2)
ax.plot(hist.index, val_hist, label='Loss Validación', color='#dc2626', linewidth=2, linestyle='--')
ax.axvline(ep_conv, color='#16a34a', linestyle='-.', label=f'Convergencia práctica ({ep_conv})')
ax.axvline(best_ep, color='gray', linestyle=':', label=f'Mejor época ({best_ep})')
ax.set_title(f'Evolución de la pérdida {VERSION.upper()} (N_BASES = {N_BASES} | {N_PUNTOS} puntos)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Época', fontsize=10)
ax.set_ylabel('Pérdida compuesta', fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
if lr_col:
    ax_lr = ax.twinx()
    ax_lr.plot(hist.index, hist[lr_col], color='#6b7280', linewidth=1, alpha=0.7, label='Learning rate')
    ax_lr.set_yscale('log')
    ax_lr.set_ylabel('Learning rate', fontsize=9)
    l1, n1 = ax.get_legend_handles_labels()
    l2, n2 = ax_lr.get_legend_handles_labels()
    ax.legend(l1 + l2, n1 + n2, fontsize=9)
else:
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"evolucion_perdida_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Célula 8b — Evaluación base en test (la reutilizan 9, 12, 12b y 13)

In [ ]:
# ==============================================================================
# CÉLULA 8b: EVALUACIÓN BASE SOBRE EL CONJUNTO DE TEST (la usan las Células 9, 12 y 13)
# ==============================================================================
"""
Un único lugar para la inferencia sobre test:
  • coef_crudo : salida de la red desestandarizada (sin proyectar).
  • coef_rn    : proyección de coef_crudo sobre la caja [0, MAX_COEF] (paso explícito del método).
  • coef_lsq   : óptimo acotado de la curva de ENTRADA (techo alcanzable con las bases).
  • coef_opt   : etiqueta (= coef_lsq cuando ETIQUETA_LSQ_EN_TODAS = True).
"""
import time
import numpy as np

N_CRONO_LSQ = 300   # curvas para cronometrar lsq_linear


def evaluar_test():
    y_pred_norm = model.predict(X_test, batch_size=1024, verbose=0)
    coef_crudo = sy.inverse_transform(y_pred_norm)
    coef_rn = np.clip(coef_crudo, COEF_MIN, MAX_COEF)
    coef_opt = sy.inverse_transform(y_test)
    b_in = Y_raw[idx_test]
    coef_lsq = coef_opt.copy() if ETIQUETA_LSQ_EN_TODAS else np.array([lsq_acotado(A_base, y) for y in b_in])
    return {
        "coef_crudo": coef_crudo,
        "coef_rn": coef_rn,                              # (n_test, N_BASES) predicho y proyectado
        "coef_opt": coef_opt,                            # (n_test, N_BASES) etiqueta
        "coef_lsq": coef_lsq,                            # (n_test, N_BASES) óptimo de la entrada
        "b_rn": coef_rn @ A_base.T,                      # curva reconstruida por la red
        "b_opt": coef_opt @ A_base.T,                    # curva reconstruida con la etiqueta
        "b_lsq": coef_lsq @ A_base.T,                    # mejor reconstrucción posible de la entrada
        "b_in": b_in,                                    # curva REAL de entrada a la red
        "fam": fam_test,
        "idx": idx_test,
    }


EVAL = evaluar_test()
n_test = EVAL["coef_rn"].shape[0]
neg = EVAL["coef_crudo"] < COEF_MIN
sobre = EVAL["coef_crudo"] > MAX_COEF
print(f"✅ Evaluación base lista: {n_test} curvas de test, {EVAL['coef_rn'].shape[1]} coeficientes.")
print(f"   • Predicciones crudas fuera de la caja (antes de proyectar): "
      f"negativas {neg.mean():.2%} de los coeficientes ({neg.any(axis=1).mean():.2%} de las curvas, "
      f"mínimo {EVAL['coef_crudo'].min():.4f}) | > MAX_COEF {sobre.mean():.2%}")

# --- Cronometraje justo: red en lote vs. lsq_linear curva a curva -----------------------------
Y_crono = Y_raw[idx_test]
t0 = time.perf_counter()
_ = np.clip(sy.inverse_transform(model.predict(sx.transform(safe_log_transform(Y_crono)),
                                               batch_size=1024, verbose=0)), COEF_MIN, MAX_COEF)
t_rn = (time.perf_counter() - t0) / len(Y_crono)
t0 = time.perf_counter()
for y in Y_crono[:N_CRONO_LSQ]:
    lsq_acotado(A_base, y)
t_lsq = (time.perf_counter() - t0) / N_CRONO_LSQ
TIEMPOS = {"rn_ms_por_curva": t_rn * 1e3, "lsq_ms_por_curva": t_lsq * 1e3}
print(f"   • Tiempo por curva (incluye log + escalado): red en lote {t_rn*1e3:.4f} ms | "
      f"lsq_linear {t_lsq*1e3:.3f} ms → factor {t_lsq / t_rn:.1f}×")

## Célula 9 — Reconstrucción física en test y fronteras analíticas

In [ ]:
# ==============================================================================
# CÉLULA 9: RECONSTRUCCIÓN FÍSICA EN TEST (UNA MUESTRA ALEATORIA POR FAMILIA)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt

x_mesh = X_MESH_999
y1_full = y1_recortada(x_mesh)
y2_full = y2_recortada(x_mesh)

rng_plot = np.random.default_rng(SEED)
idx_muestras = [int(rng_plot.choice(np.where(EVAL["fam"] == f)[0])) for f in sorted(FAMILIAS)]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, s in zip(axes.flatten(), idx_muestras):
    b_in, b_rn, b_lsq = EVAL["b_in"][s], EVAL["b_rn"][s], EVAL["b_lsq"][s]
    c_pred, c_true = EVAL["coef_rn"][s], EVAL["coef_opt"][s]
    m_rn, m_lsq = metricas_curva(b_in, b_rn), metricas_curva(b_in, b_lsq)

    ax.plot(x_mesh, b_in, 'k-', lw=2.2, label='Curva de entrada ($P_{real}$)')
    ax.plot(x_mesh, b_lsq, 'b:', lw=1.8, label='Óptimo acotado (lsq_linear)')
    ax.plot(x_mesh, b_rn, 'r--', lw=2.0, label=f'Reconstrucción red {VERSION.upper()}')
    ax.plot(x_mesh, y1_full, 'g:', alpha=0.6, lw=1.2, label='Frontera Tabla 3 (≥0)')
    ax.plot(x_mesh, y2_full, 'c:', alpha=0.6, lw=1.2, label='Frontera Tabla 6 (≥0)')
    ax.set_ylim(bottom=-0.03 * max(b_in.max(), 1.0), top=1.15 * max(b_in.max(), 1.0))   # escala por panel

    coef_str = ", ".join(f"$c_{i+1}$={c_pred[i]:.2f}" for i in range(N_BASES))
    true_str = ", ".join(f"{c_true[i]:.2f}" for i in range(N_BASES))
    ax.set_title(f"Test #{s} — {FAMILIAS[int(EVAL['fam'][s])]} | $R^2$ red={m_rn['R2']:.5f} · óptimo={m_lsq['R2']:.5f}\n"
                 f"pred [{coef_str}]\nreal [{true_str}]", fontsize=9, fontweight='bold')
    ax.set_xlabel(r"Estiramiento arterial $\lambda_\theta$", fontsize=9)
    ax.set_ylabel("Presión intraluminal $P_i$ [kPa]", fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=7)

plt.suptitle(f"Reconstrucción física red {VERSION.upper()} — dominio [{X_MIN_DOMINIO}, {X_MAX_DOMINIO}]",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"reconstruccion_curvas_test_{VERSION}.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# --- Inferencia en las fronteras analíticas (la comparación con el óptimo está en la Célula 10) ---
print("\n=== INFERENCIA EN LAS FRONTERAS ANALÍTICAS (Tabla 3 y Tabla 6) ===")
for nombre, y_fr in [("Tabla 3", y1_full), ("Tabla 6", y2_full)]:
    Xn = sx.transform(safe_log_transform(y_fr).reshape(1, -1))
    c_crudo = sy.inverse_transform(model.predict(Xn, verbose=0))[0].astype(np.float64)
    c_fr = np.clip(c_crudo, COEF_MIN, MAX_COEF)
    print(f"✅ Frontera {nombre}: c proyectado = {np.round(c_fr, 4)} | crudo = {np.round(c_crudo, 4)}")

## Célula 10 — Fronteras y1 (Tabla 3) / y2 (Tabla 6): red vs. óptimo lsq_linear con verificación KKT

In [ ]:
# ==============================================================================
# CÉLULA 10: FRONTERAS y1(x) [TABLA 3] E y2(x) [TABLA 6] — RED vs. ÓPTIMO lsq_linear (con verificación KKT)
# ==============================================================================
"""
Problema: min_c ½‖A c − y‖²  s.a.  0 <= c_i <= MAX_COEF.
  • Óptimo: lsq_linear (mínimos cuadrados acotados), la MISMA referencia que usan las etiquetas
    del dataset (Célula 4) y el test (Células 8b y 12).
  • Verificación KKT post-hoc sobre la solución de lsq_linear, a partir de g = Aᵀ(A c* − y):
      cota inferior activa → μ_i = g_i >= 0 ; cota superior activa → ν_i = −g_i >= 0 ;
      componente libre → g_i ≈ 0 (estacionariedad).
    Las cotas activas se cruzan con el active_mask que devuelve lsq_linear.
  • Tiempos: mediana de varias corridas de lsq_linear vs. inferencia de la red (1 curva).
Las fronteras se evalúan recortadas a presión >= 0. Tabla 3 y Tabla 6 se cruzan en x ≈ 0.996.
"""
import time
import numpy as np
from scipy.optimize import lsq_linear

VENTANA_EVAL = (X_MIN_DOMINIO, X_MAX_DOMINIO)
MASK_EVAL = (X_MESH_999 >= VENTANA_EVAL[0]) & (X_MESH_999 <= VENTANA_EVAL[1])
x_eval = X_MESH_999[MASK_EVAL]
A_eval = A_base[MASK_EVAL, :]
N_REP_TIEMPO = 50
TOL_ACTIVA = 1e-6


def _mediana_tiempo(fn, n=N_REP_TIEMPO):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def verificacion_kkt(A, y, c, active_mask):
    """Multiplicadores de Lagrange de las cotas y chequeos KKT para la solución c de lsq_linear."""
    g = A.T @ (A @ c - y)
    inf = c <= COEF_MIN + TOL_ACTIVA
    sup = c >= MAX_COEF - TOL_ACTIVA
    libre = ~(inf | sup)
    mu, nu = np.where(inf, g, 0.0), np.where(sup, -g, 0.0)
    escala = max(float(np.linalg.norm(A.T @ y)), 1e-12)
    return {"mu_inf": mu, "nu_sup": nu,
            "activas_inf": np.where(inf)[0] + 1, "activas_sup": np.where(sup)[0] + 1,
            "estacionariedad_rel": (float(np.max(np.abs(g[libre]))) if libre.any() else 0.0) / escala,
            "signos_ok": bool(np.all(mu >= -1e-8 * escala) and np.all(nu >= -1e-8 * escala)),
            "coincide_active_mask": bool(np.array_equal(inf, active_mask == -1) and np.array_equal(sup, active_mask == 1))}


def evaluar_frontera(y_full, nombre):
    y_ac = y_full[MASK_EVAL]

    res = lsq_linear(A_eval, y_ac, bounds=(COEF_MIN, MAX_COEF))
    coef_opt = res.x
    t_lsq = _mediana_tiempo(lambda: lsq_acotado(A_eval, y_ac))

    def _inferir():
        Xn = sx.transform(safe_log_transform(y_full).reshape(1, -1))
        return sy.inverse_transform(model(Xn, training=False).numpy()).flatten().astype(np.float64)

    coef_crudo = _inferir()
    t_rn = _mediana_tiempo(_inferir)
    coef_pred = np.clip(coef_crudo, COEF_MIN, MAX_COEF)

    b_rn = (A_base @ coef_pred)[MASK_EVAL]
    b_opt = A_eval @ coef_opt
    return {"nombre": nombre, "y": y_ac, "coef_pred": coef_pred, "coef_crudo": coef_crudo,
            "coef_opt": coef_opt, "b_rn": b_rn, "b_opt": b_opt, "t_lsq": t_lsq, "t_rn": t_rn,
            "kkt": verificacion_kkt(A_eval, y_ac, coef_opt, res.active_mask),
            "m_rn": metricas_curva(y_ac, b_rn), "m_opt": metricas_curva(y_ac, b_opt)}


def reportar_frontera(r):
    k = r["kkt"]
    print("\n" + "=" * 92)
    print(f"EVALUACIÓN EN DOMINIO {VENTANA_EVAL}: {r['nombre']}  (N = {N_BASES})")
    print("=" * 92)
    print(f"Coeficientes óptimos (lsq_linear): {np.round(r['coef_opt'], 4)}")
    print(f"Coeficientes red                 : {np.round(r['coef_pred'], 4)}   (crudos: {np.round(r['coef_crudo'], 4)})")
    print(f"Cotas activas en el óptimo: inferior {k['activas_inf'].tolist() or '—'} | superior {k['activas_sup'].tolist() or '—'}"
          f" | coincide con active_mask de lsq_linear: {k['coincide_active_mask']}")
    print(f"Multiplicadores KKT  μ (c >= 0): {np.round(k['mu_inf'], 4)} | ν (c <= {MAX_COEF:g}): {np.round(k['nu_sup'], 4)}")
    print(f"KKT: estacionariedad relativa en componentes libres = {k['estacionariedad_rel']:.2e} | "
          f"signos de los multiplicadores correctos: {k['signos_ok']}")
    print(f"Tiempo (mediana de {N_REP_TIEMPO}): lsq_linear {r['t_lsq']*1e3:.3f} ms | red (1 curva) {r['t_rn']*1e3:.3f} ms")
    print("-" * 92)
    print(f"{'':<12} {'R²':>10} {'RMSE [kPa]':>11} {'MaxAE [kPa]':>12} {'NRMSE':>9} {'MaxAE_rel':>10}")
    for etiqueta, m in [("Red", r["m_rn"]), ("Óptimo lsq", r["m_opt"])]:
        print(f"{etiqueta:<12} {m['R2']:10.6f} {m['RMSE']:11.4f} {m['MaxAE']:12.4f} "
              f"{m['NRMSE']:8.4f}% {m['MaxAE_rel']:9.3f}%")
    exceso = (r["m_rn"]["RMSE"] / r["m_opt"]["RMSE"] - 1.0) * 100.0
    print(f"Exceso de RMSE de la red sobre el óptimo: {exceso:+.2f}%")
    print("=" * 92)


RES_Y1 = evaluar_frontera(y1_recortada(X_MESH_999), "y1(x) [Tabla 3 - Normotensos, recortada ≥0]")
RES_Y2 = evaluar_frontera(y2_recortada(X_MESH_999), "y2(x) [Tabla 6 - Hipertensos, recortada ≥0]")
reportar_frontera(RES_Y1)
reportar_frontera(RES_Y2)

## Célula 11 — Gráficos de reconstrucción y residuos en y1/y2

In [ ]:
# ==============================================================================
# CÉLULA 11: VISUALIZACIÓN DE RECONSTRUCCIÓN Y RESIDUOS EN y1(x) E y2(x)
# ==============================================================================
"""
Los picos de los residuos caen en los nudos de las rampas: es la firma de aproximar una curva
convexa con una función lineal a trozos (límite de la base, no de la red).
"""
import matplotlib.pyplot as plt

nudos = PARAMS_BASES["knots"] if "PARAMS_BASES" in globals() else []

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for fila, (res, titulo) in enumerate([(RES_Y1, "y1(x) [Tabla 3 - Normotensos]"),
                                      (RES_Y2, "y2(x) [Tabla 6 - Hipertensos]")]):
    ax_a, ax_r = axes[fila]
    corto = titulo.split(' ')[0]
    ax_a.plot(x_eval, res["y"], "k-", linewidth=2.5, label=f"{corto} real")
    ax_a.plot(x_eval, res["b_rn"], "r--", linewidth=2, label=f"Predicción red {VERSION.upper()}")
    ax_a.plot(x_eval, res["b_opt"], "b:", linewidth=1.5, label="Óptimo lsq_linear")
    ax_a.set_ylabel(r"Presión $P_i$ [kPa]", fontsize=10)
    ax_a.set_title(f"Ajuste en {titulo}\nNRMSE red {res['m_rn']['NRMSE']:.2f}% (óptimo {res['m_opt']['NRMSE']:.2f}%) | "
                   f"MaxAE red {res['m_rn']['MaxAE']:.2f} kPa", fontsize=10, fontweight="bold")
    ax_a.legend(fontsize=9)
    ax_a.grid(True, alpha=0.3)

    ax_r.plot(x_eval, res["y"] - res["b_rn"], "r-", linewidth=1.5, label=f"Residuo red {VERSION.upper()}")
    ax_r.plot(x_eval, res["y"] - res["b_opt"], "b--", linewidth=1.5, label="Residuo óptimo lsq_linear")
    for k in nudos:
        ax_r.axvline(k, color="gray", linestyle="-.", alpha=0.5, linewidth=1)
    ax_r.axhline(0, color="black", linestyle=":", alpha=0.6)
    ax_r.set_ylabel("Residuo (real − predicho) [kPa]", fontsize=10)
    ax_r.set_title(f"Residuos en {corto} (líneas grises: nudos de las rampas)", fontsize=11, fontweight="bold")
    ax_r.legend(fontsize=9)
    ax_r.grid(True, alpha=0.3)

for ax in axes[1]:
    ax.set_xlabel(r"Estiramiento arterial $\lambda_\theta$", fontsize=10)

plt.tight_layout()
plt.savefig(f"Grafico_Reconstruccion_Fronteras_y1_y2_{VERSION}.png", dpi=300)
plt.show()
plt.close()

## Célula 12 — Evaluación en test: coeficientes, brecha red vs. óptimo, peores casos

In [ ]:
# ==============================================================================
# CÉLULA 12: EVALUACIÓN EN TEST — COEFICIENTES, BRECHA RED vs. ÓPTIMO, PEORES CASOS
# ==============================================================================
"""
Todas las métricas de curva comparan contra la curva de ENTRADA (la que ve la red):
  • red     : A·c_red      (coef_rn, proyectado sobre la caja)
  • óptimo  : A·c_lsq      (techo alcanzable con las bases sobre ESA curva)
La brecha red − óptimo es lo atribuible a la red; el óptimo mide el piso de las bases.
NRMSE y MaxAE_rel se calculan por curva con su propio rango (Célula 3).
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"📊 Evaluando la red {VERSION.upper()} en el conjunto de prueba (Test)...\n")

c_rn, c_opt, c_crudo = EVAL["coef_rn"], EVAL["coef_opt"], EVAL["coef_crudo"]
b_in, b_rn, b_lsq = EVAL["b_in"], EVAL["b_rn"], EVAL["b_lsq"]
fam_t = EVAL["fam"]
N_COEFS = c_opt.shape[1]
M_RN = metricas_lote(b_in, b_rn)
M_OPT = metricas_lote(b_in, b_lsq)
m_env = fam_t >= 2      # familias de envolvente = dominio objetivo


# --- 1. Coeficientes: global y restringido a las familias de envolvente --------------------------
def _tabla_coef(mask, titulo):
    print("=" * 80)
    print(f"🎯 PRECISIÓN DE COEFICIENTES — {titulo} (n = {mask.sum()})")
    print("=" * 80)
    print(f"{'Coef.':<8} | {'R²':>10} | {'MAE':>10} | {'RMSE':>10} | {'rango etiqueta':>16}")
    print("-" * 66)
    out = []
    for i in range(N_COEFS):
        yt, yp = c_opt[mask, i], c_rn[mask, i]
        fila = (r2_score(yt, yp), mean_absolute_error(yt, yp), np.sqrt(mean_squared_error(yt, yp)))
        out.append(fila)
        print(f"c{i+1:<7} | {fila[0]:10.6f} | {fila[1]:10.6f} | {fila[2]:10.6f} | [{yt.min():5.2f}, {yt.max():5.2f}]")
    out = np.array(out)
    print("-" * 66)
    print(f"{'PROMEDIO':<8} | {out[:, 0].mean():10.6f} | {out[:, 1].mean():10.6f} | {out[:, 2].mean():10.6f}")
    return out


COEF_GLOBAL = _tabla_coef(np.ones_like(fam_t, dtype=bool), "TODAS LAS FAMILIAS")
COEF_ENV = _tabla_coef(m_env, "SOLO FAMILIAS DE ENVOLVENTE (dominio objetivo)")
print("   (el R² global se infla porque las familias 0 y 1 estiran el rango de las etiquetas a [0, MAX_COEF])")

neg = c_crudo < COEF_MIN
print(f"\n🔒 Predicciones crudas negativas: {neg.mean():.2%} de los coeficientes | "
      f"> MAX_COEF: {(c_crudo > MAX_COEF).mean():.2%} (se proyectan sobre la caja)")

# --- 2. Reconstrucción por familia: red vs. óptimo sobre la misma curva --------------------------
print("\n" + "=" * 118)
print("🧬 RECONSTRUCCIÓN POR FAMILIA (contra la curva de entrada)")
print("=" * 118)
print(f"{'Familia':<15} {'n':>5} | {'R² óptimo':>10} {'R² red':>9} {'med. red':>9} {'mín. red':>9} {'(test #)':>9} | "
      f"{'NRMSE ópt.':>10} {'NRMSE red':>10} {'p95 red':>8} | {'Exceso':>8} | {'MaxAE_rel p95':>13}")
print("-" * 118)
RESUMEN_FAM = []
for f, nombre in FAMILIAS.items():
    m = fam_t == f
    i_peor = int(np.where(m)[0][np.nanargmin(M_RN["R2"][m])])
    exceso = (np.nanmean(M_RN["NRMSE"][m]) / np.nanmean(M_OPT["NRMSE"][m]) - 1.0) * 100.0
    fila = dict(familia=nombre, n=int(m.sum()),
                R2_opt=np.nanmean(M_OPT["R2"][m]), R2_red=np.nanmean(M_RN["R2"][m]),
                R2_red_mediana=np.nanmedian(M_RN["R2"][m]), R2_red_min=np.nanmin(M_RN["R2"][m]),
                R2_opt_en_peor=M_OPT["R2"][i_peor], idx_peor=i_peor,
                NRMSE_opt=np.nanmean(M_OPT["NRMSE"][m]), NRMSE_red=np.nanmean(M_RN["NRMSE"][m]),
                NRMSE_red_mediana=np.nanmedian(M_RN["NRMSE"][m]), NRMSE_red_p95=np.nanpercentile(M_RN["NRMSE"][m], 95),
                exceso_NRMSE_pct=exceso, MaxAE_rel_p95=np.nanpercentile(M_RN["MaxAE_rel"][m], 95),
                MaxAE_kPa_max=np.nanmax(M_RN["MaxAE"][m]),
                MAE_coef=np.abs(c_opt[m] - c_rn[m]).mean(axis=0))
    RESUMEN_FAM.append(fila)
    print(f"{nombre:<15} {fila['n']:>5} | {fila['R2_opt']:10.5f} {fila['R2_red']:9.5f} {fila['R2_red_mediana']:9.5f} "
          f"{fila['R2_red_min']:9.5f} {i_peor:>9d} | {fila['NRMSE_opt']:9.3f}% {fila['NRMSE_red']:9.3f}% "
          f"{fila['NRMSE_red_p95']:7.3f}% | {exceso:+7.1f}% | {fila['MaxAE_rel_p95']:12.3f}%")
print("-" * 118)
print("Exceso = NRMSE medio de la red / NRMSE medio del óptimo − 1 (lo que pierde la red frente al techo de las bases).")
print("\nMAE de coeficientes por familia:")
for fila in RESUMEN_FAM:
    print(f"   • {fila['familia']:<15}: {np.round(fila['MAE_coef'], 4)} | peor caso: test #{fila['idx_peor']} "
          f"(R² red {fila['R2_red_min']:.5f} vs. óptimo {fila['R2_opt_en_peor']:.5f}) | MaxAE máx. {fila['MaxAE_kPa_max']:.2f} kPa")

# --- 3. Persistencia de métricas por curva (para el artículo) ------------------------------------
DF_TEST = pd.DataFrame({"idx_test": np.arange(len(fam_t)), "idx_dataset": EVAL["idx"],
                        "familia": [FAMILIAS[int(f)] for f in fam_t]})
for k in ("R2", "NRMSE", "MaxAE", "MaxAE_rel"):
    DF_TEST[f"{k}_red"], DF_TEST[f"{k}_opt"] = M_RN[k], M_OPT[k]
for i in range(N_COEFS):
    DF_TEST[f"c{i+1}_etiqueta"], DF_TEST[f"c{i+1}_red"], DF_TEST[f"c{i+1}_crudo"] = c_opt[:, i], c_rn[:, i], c_crudo[:, i]
DF_TEST.to_csv(f"metricas_test_por_curva_{VERSION}.csv", index=False)
pd.DataFrame([{k: v for k, v in r.items() if k != "MAE_coef"} for r in RESUMEN_FAM]).to_csv(
    f"resumen_test_por_familia_{VERSION}.csv", index=False)
print(f"\n💾 'metricas_test_por_curva_{VERSION}.csv' y 'resumen_test_por_familia_{VERSION}.csv' guardados.")

# --- Figura 1: regresión 1:1 de coeficientes ------------------------------------------------------
colores = {0: "#d62728", 1: "#2ca02c", 2: "#1f77b4", 3: "#ff7f0e"}
n_cols = 2
n_rows = int(np.ceil(N_COEFS / n_cols))
fig1, axes1 = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows), squeeze=False)
axes1 = axes1.flatten()
for i in range(N_COEFS):
    ax = axes1[i]
    for f, nombre in FAMILIAS.items():
        m = fam_t == f
        ax.scatter(c_opt[m, i], c_rn[m, i], alpha=0.3, color=colores[f], edgecolors="none", s=15, label=nombre)
    lims = [min(c_opt[:, i].min(), c_rn[:, i].min()), max(c_opt[:, i].max(), c_rn[:, i].max())]
    ax.plot(lims, lims, "k--", label="Ideal 1:1", linewidth=1.8)
    ax.set_title(f"Coeficiente c{i+1}\n$R^2$ global = {COEF_GLOBAL[i, 0]:.4f} | $R^2$ envolvente = {COEF_ENV[i, 0]:.4f}",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Real (etiqueta)", fontsize=10)
    ax.set_ylabel(f"Predicho (red {VERSION.upper()})", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=9)
for j in range(N_COEFS, len(axes1)):
    axes1[j].axis("off")
plt.tight_layout()
plt.savefig(f"diagnostico_coeficientes_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Figura 2: log10(1 − R²) por familia y residuos relativos ------------------------------------
fig2, (ax_r2, ax_res) = plt.subplots(1, 2, figsize=(15, 5))
log_err = np.log10(np.clip(1.0 - M_RN["R2"], 1e-8, None))
bins = np.linspace(np.nanmin(log_err), np.nanmax(log_err), 50)
for f, nombre in FAMILIAS.items():
    m = fam_t == f
    ax_r2.hist(log_err[m], bins=bins, alpha=0.55, color=colores[f], label=nombre)
ax_r2.set_yscale("log")
ax_r2.set_xlabel(r"$\log_{10}(1 - R^2)$ por curva (red vs. entrada) — más a la izquierda es mejor", fontsize=10)
ax_r2.set_ylabel("Frecuencia (escala log)", fontsize=10)
ax_r2.set_title("Distribución del error de reconstrucción por familia", fontweight="bold")
ax_r2.grid(True, linestyle="--", alpha=0.5)
ax_r2.legend(fontsize=9)

y_max_curva = np.maximum(b_in.max(axis=1, keepdims=True), 1.0)
res_red_rel = ((b_lsq - b_rn) / y_max_curva * 100.0)       # error atribuible a la red (% de y_max)
res_tot_rel = ((b_in - b_rn) / y_max_curva * 100.0)        # error total contra la entrada
for datos, color, etiqueta in [(res_tot_rel, "#9467bd", "entrada − red (total)"),
                               (res_red_rel, "#ef4444", "óptimo − red (atribuible a la red)")]:
    ax_res.hist(datos.flatten(), bins=200, density=True, alpha=0.5, color=color, label=etiqueta)
ax_res.set_yscale("log")
ax_res.axvline(0, color="black", linestyle="--", alpha=0.7)
ax_res.set_xlabel("Residuo relativo [% del y_max de cada curva]", fontsize=10)
ax_res.set_ylabel("Densidad (escala log)", fontsize=10)
ax_res.set_title(f"Residuos relativos {VERSION.upper()} (todas las familias)", fontsize=11, fontweight="bold")
ax_res.legend(fontsize=9)
ax_res.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"distribucion_residuos_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nPercentiles de |residuo relativo| [% de y_max] (por punto):")
print(f"{'Familia':<15} | {'tipo':<12} | {'p50':>7} {'p95':>7} {'p99':>7} {'máx':>7}")
for f, nombre in FAMILIAS.items():
    m = fam_t == f
    for tipo, datos in [("total", res_tot_rel), ("red−óptimo", res_red_rel)]:
        q = np.percentile(np.abs(datos[m]), [50, 95, 99, 100])
        print(f"{nombre:<15} | {tipo:<12} | {q[0]:7.3f} {q[1]:7.3f} {q[2]:7.3f} {q[3]:7.3f}")

# --- Figura 3: peor caso de cada familia ------------------------------------------------------------
fig3, axes3 = plt.subplots(1, len(FAMILIAS), figsize=(5.2 * len(FAMILIAS), 4.5))
for ax, fila in zip(np.atleast_1d(axes3), RESUMEN_FAM):
    s = fila["idx_peor"]
    ax.plot(X_MESH_999, b_in[s], "k-", lw=2, label="Entrada")
    ax.plot(X_MESH_999, b_lsq[s], "b:", lw=1.8, label=f"Óptimo ($R^2$={M_OPT['R2'][s]:.4f})")
    ax.plot(X_MESH_999, b_rn[s], "r--", lw=1.8, label=f"Red ($R^2$={M_RN['R2'][s]:.4f})")
    ax.set_title(f"Peor caso — {fila['familia']} (test #{s})", fontsize=10, fontweight="bold")
    ax.set_xlabel(r"Estiramiento $\lambda_\theta$", fontsize=9)
    ax.set_ylabel("Presión [kPa]", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(f"peores_casos_test_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Célula 12 ejecutada correctamente.")

## Célula 12b — Generalización fuera de distribución (perfiles α(x) no vistos)

In [ ]:
# ==============================================================================
# CÉLULA 12b: GENERALIZACIÓN FUERA DE DISTRIBUCIÓN (perfiles alpha(x) NUNCA vistos)
# ==============================================================================
"""
Perfiles alpha(x) que el generador de la Célula 4 NO produce:
  • doble transición (0→1→0 y 1→0→1)
  • sinusoidal de amplitud grande (1 a 3 ciclos)
Estas curvas NO se convexifican: pueden ser no monótonas o no convexas, es decir, salir de la
clase que las rampas pueden representar. Se informa qué fracción lo es, porque eso define si
el test mide generalización DENTRO de la clase de curvas J o extrapolación fuera de ella.
La métrica relevante es la brecha red − óptimo lsq_linear sobre la misma curva. Fuera de
distribución el MAE de coeficientes es engañoso (la pérdida es plana cerca del óptimo cuando
el residuo de las bases es grande): se informa solo como referencia.
"""
import numpy as np

N_OOD = 300
RUIDO_OOD = 0.01
rng_ood = np.random.default_rng(SEED_DATASET + 1)
lo_e, hi_e = envolvente_inferior(X_MESH_999), envolvente_superior(X_MESH_999)
t_ood = (X_MESH_999 - X_MIN_DOMINIO) / (X_MAX_DOMINIO - X_MIN_DOMINIO)


def _sig(x0, w):
    return 1.0 / (1.0 + np.exp(-np.clip((X_MESH_999 - x0) / w, -60.0, 60.0)))


def alpha_doble_transicion():
    sube = rng_ood.random() < 0.5
    a0, a1, a2 = ((rng_ood.uniform(-0.05, 0.15), rng_ood.uniform(0.85, 1.05), rng_ood.uniform(-0.05, 0.15)) if sube
                  else (rng_ood.uniform(0.85, 1.05), rng_ood.uniform(-0.05, 0.15), rng_ood.uniform(0.85, 1.05)))
    xa = rng_ood.uniform(0.88, 1.05)
    xb = xa + rng_ood.uniform(0.04, 0.12)
    wa, wb = 10.0 ** rng_ood.uniform(-2.7, -1.3, size=2)
    return a0 + (a1 - a0) * _sig(xa, wa) + (a2 - a1) * _sig(xb, wb)


def alpha_sinusoidal():
    f = rng_ood.uniform(1.0, 3.0)
    return 0.5 + 0.45 * np.sin(2.0 * np.pi * f * t_ood + rng_ood.uniform(0, 2.0 * np.pi))


def generar_ood(fn_alpha, n):
    Y = np.zeros((n, N_PUNTOS_MALLA))
    forma = np.zeros((n, 2), dtype=bool)          # (monótona, convexa) de la curva limpia
    for i in range(n):
        y0 = np.maximum(lo_e + np.clip(fn_alpha(), -0.05, 1.05) * (hi_e - lo_e), PRESION_MIN_KPA)
        forma[i] = diagnostico_forma(y0)
        y = y0 + rng_ood.normal(0.0, RUIDO_OOD, size=N_PUNTOS_MALLA) * np.abs(y0)
        Y[i] = np.maximum(y, PRESION_MIN_KPA)
    return Y, forma


def evaluar_ood(nombre, Y, forma):
    c_opt = np.array([lsq_acotado(A_base, y) for y in Y])
    Xn = sx.transform(safe_log_transform(Y))
    c_rn = np.clip(sy.inverse_transform(model.predict(Xn, verbose=0)), COEF_MIN, MAX_COEF)
    m_rn, m_opt = metricas_lote(Y, c_rn @ A_base.T), metricas_lote(Y, c_opt @ A_base.T)
    i_peor = int(np.nanargmin(m_rn["R2"]))
    return {"nombre": nombre, "m_rn": m_rn, "m_opt": m_opt, "i_peor": i_peor,
            "frac_monotona": forma[:, 0].mean(), "frac_mono_convexa": (forma[:, 0] & forma[:, 1]).mean(),
            "mae_c": np.abs(c_rn - c_opt).mean(axis=0)}


print("🧪 Test fuera de distribución (perfiles alpha(x) no vistos en el entrenamiento)\n")
print(f"{'Familia OOD':<18} | {'R² red':>8} {'R² ópt.':>8} | {'NRMSE red':>9} {'NRMSE ópt.':>10} {'Exceso':>8} | "
      f"{'peor R² red':>11} {'ópt. en esa':>11} | {'monót.':>7} {'mon.+conv.':>10}")
print("-" * 118)
RES_OOD = []
for nombre, fn in [("doble_transicion", alpha_doble_transicion), ("sinusoidal", alpha_sinusoidal)]:
    r = evaluar_ood(nombre, *generar_ood(fn, N_OOD))
    RES_OOD.append(r)
    exceso = (np.nanmean(r["m_rn"]["NRMSE"]) / np.nanmean(r["m_opt"]["NRMSE"]) - 1.0) * 100.0
    r["exceso_NRMSE_pct"] = exceso
    print(f"{nombre:<18} | {np.nanmean(r['m_rn']['R2']):8.5f} {np.nanmean(r['m_opt']['R2']):8.5f} | "
          f"{np.nanmean(r['m_rn']['NRMSE']):8.3f}% {np.nanmean(r['m_opt']['NRMSE']):9.3f}% {exceso:+7.1f}% | "
          f"{r['m_rn']['R2'][r['i_peor']]:11.5f} {r['m_opt']['R2'][r['i_peor']]:11.5f} | "
          f"{r['frac_monotona']:6.1%} {r['frac_mono_convexa']:9.1%}")
print("-" * 118)
for r in RES_OOD:
    print(f" • {r['nombre']:<18} MAE de coeficientes c1..c{N_BASES} (referencia): {np.round(r['mae_c'], 4)}")
print("\nLectura: la brecha red − óptimo es lo que la red pierde por generalizar; el óptimo es el techo de las bases.")
print("Las curvas no monótonas o no convexas quedan fuera de la clase representable con rampas y c >= 0.")

## Célula 13 — Reconstrucción de muestras aleatorias (escala por panel)

In [ ]:
# ==============================================================================
# CÉLULA 13: RECONSTRUCCIÓN FÍSICA DE MUESTRAS ALEATORIAS (2 POR FAMILIA)
# ==============================================================================
"""
Escala vertical por panel (en V12 era fija en 0–200 kPa y aplastaba las familias de envolvente).
Los peores casos de cada familia están en la Figura 3 de la Célula 12.
"""
import numpy as np
import matplotlib.pyplot as plt

rng_plot = np.random.default_rng(SEED)
indices_muestras = []
for f in sorted(FAMILIAS):
    cand = np.where(EVAL["fam"] == f)[0]
    indices_muestras += [int(i) for i in rng_plot.choice(cand, size=min(2, len(cand)), replace=False)]

fig, axes = plt.subplots(2, 4, figsize=(21, 10))
for ax, s in zip(axes.flatten(), indices_muestras):
    b_in, b_pred, b_lsq = EVAL["b_in"][s], EVAL["b_rn"][s], EVAL["b_lsq"][s]
    m_rn, m_lsq = metricas_curva(b_in, b_pred), metricas_curva(b_in, b_lsq)
    ax.plot(X_MESH_999, b_in, "k-", linewidth=2.0, label="Curva de entrada", alpha=0.85)
    ax.plot(X_MESH_999, b_lsq, "b:", linewidth=1.6, label="Óptimo acotado", alpha=0.9)
    ax.plot(X_MESH_999, b_pred, "r--", linewidth=1.8, label=f"Predicción red {VERSION.upper()}", alpha=0.9)
    ax.set_title(f"Muestra #{s} — {FAMILIAS[int(EVAL['fam'][s])]}\n"
                 f"$R^2$ red {m_rn['R2']:.5f} (óptimo {m_lsq['R2']:.5f}) | MaxAE {m_rn['MaxAE']:.3f} kPa",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel(r"Estiramiento $\lambda$", fontsize=9)
    ax.set_ylabel(r"Presión $P(\lambda)$ [kPa]", fontsize=9)
    ax.set_xlim(X_MIN_DOMINIO, X_MAX_DOMINIO)
    ax.set_ylim(-0.03 * max(b_in.max(), 1.0), 1.15 * max(b_in.max(), 1.0))
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=8)

plt.suptitle(f"Reconstrucción física de muestras de test — red {VERSION.upper()}",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"reconstruccion_ejemplos_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"✅ Célula 13 ejecutada. Ventana: λ ∈ [{X_MIN_DOMINIO}, {X_MAX_DOMINIO}] | escala vertical por panel.")

## Célula 14 — Evaluador universal de Curvas J (sin piso de envolvente por defecto)

In [ ]:
# ==============================================================================
# CÉLULA 14: EVALUADOR UNIVERSAL DE CURVAS J (RED V13 vs. ÓPTIMO ACOTADO)
# ==============================================================================
"""
Requiere haber ejecutado las Células G y 3 (constantes, fronteras, safe_log_transform y métricas)
y haber entrenado/guardado el modelo (Célula 8). Usa los MISMOS archivos, MAX_COEF y OFFSET_LOG
que el entrenamiento. No hace relleno ni recorte silencioso de features: si hay incompatibilidad, falla.

V13: por defecto NO se aplica el piso de la envolvente. Una curva que cae fuera de la envolvente
está fuera del dominio de entrenamiento: se informa, pero no se la modifica. Si se activa el
piso, las métricas se reportan también contra la curva cruda.
"""
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from tensorflow.keras.models import load_model

for _req in ("envolvente_inferior", "envolvente_superior", "safe_log_transform", "metricas_curva",
             "lsq_acotado", "MAX_COEF", "ARCHIVO_MODELO"):
    if _req not in globals():
        raise RuntimeError(f"Falta '{_req}': ejecutá primero las Células G y 3.")

# ── 1. CONFIGURACIÓN DEL EVALUADOR ────────────────────────────────────────────────────────────
MODO_ENTRADA = "excel"                 # 'excel' o 'puntos'
TIPO_INTERPOLACION_PUNTOS = "lineal"   # 'lineal' o 'pchip'
ARCHIVO_EXCEL_CURVA_J = "Curva_J.xls"  # (.xls requiere el paquete xlrd). Hoy contiene la Base 1 (c1 = 1).
APLICAR_PISO_ENVOLVENTE = False        # True sube la curva a la envolvente inferior (altera los datos)
META_PARETO_PCT = 1.87                 # meta de error local (% del rango)
COL_OPT = "Óptimo acotado (lsq)"

# Ejemplo de curva por puntos (λ, P[kPa]) DENTRO del dominio [0.84, 1.20]
PUNTOS_USUARIO = [
    (0.84, 0.0),
    (0.95, 5.0),
    (1.05, 20.0),
    (1.15, 55.0),
    (1.20, 80.0),
]


# ── 2. CARGA DE ARCHIVOS ──────────────────────────────────────────────────────────────────────
def cargar_archivos_sistema():
    print("📂 Cargando base funcional (malla unificada)...")
    if not os.path.exists(ARCHIVO_BASES):
        raise FileNotFoundError(f"No se encontró '{ARCHIVO_BASES}'.")
    df_base = pd.read_excel(ARCHIVO_BASES, header=None)
    x_base = df_base.iloc[:, 0].values.astype(np.float64)
    A_base_ev = df_base.iloc[:, 1:].values.astype(np.float64)
    print(f"   • Malla: {len(x_base)} puntos | Bases (N_BASES): {A_base_ev.shape[1]}")

    for f in (ARCHIVO_MODELO, ARCHIVO_SCALERS):
        if not os.path.exists(f):
            raise FileNotFoundError(f"No se encontró '{f}'. Ejecutá la Célula 8 (guarda ambos archivos).")

    modelo = load_model(ARCHIVO_MODELO, compile=False)
    with open(ARCHIVO_SCALERS, "rb") as f:
        meta = pickle.load(f)
    sx_l, sy_l, offset_l = meta["scaler_x"], meta["scaler_y"], meta["offset_log"]

    if modelo.input_shape[-1] != len(x_base) or sx_l.n_features_in_ != len(x_base):
        raise ValueError(f"Incompatibilidad: modelo espera {modelo.input_shape[-1]} features, "
                         f"escalador {sx_l.n_features_in_}, malla de bases {len(x_base)}.")
    if modelo.output_shape[-1] != A_base_ev.shape[1]:
        raise ValueError(f"Incompatibilidad: el modelo predice {modelo.output_shape[-1]} coeficientes "
                         f"y las bases son {A_base_ev.shape[1]}.")
    print(f"✅ Sistema cargado ({meta.get('version', '?')}) | OFFSET_LOG = {offset_l:.4f}")
    return x_base, A_base_ev, modelo, sx_l, sy_l, offset_l


def predecir_coeficientes(modelo, sx_l, sy_l, offset_l, Y):
    """Coeficientes de la red (crudos y proyectados sobre la caja) para una o varias curvas."""
    Y = np.atleast_2d(Y)
    crudo = sy_l.inverse_transform(modelo.predict(sx_l.transform(safe_log_transform(Y, offset=offset_l)), verbose=0))
    return crudo, np.clip(crudo, COEF_MIN, MAX_COEF)


# ── 3. CURVA OBJETIVO ──────────────────────────────────────────────────────────────────────────
def generar_curva_objetivo(x_base):
    if MODO_ENTRADA == "excel":
        print(f"📖 Cargando curva desde '{ARCHIVO_EXCEL_CURVA_J}'...")
        if not os.path.exists(ARCHIVO_EXCEL_CURVA_J):
            raise FileNotFoundError(f"No se encontró '{ARCHIVO_EXCEL_CURVA_J}'.")
        df_curva = pd.read_excel(ARCHIVO_EXCEL_CURVA_J, header=None)
        x_user = df_curva.iloc[:, 0].values.astype(np.float64)
        y_user = df_curva.iloc[:, 1].values.astype(np.float64)
        orden = np.argsort(x_user)
        x_user, y_user = x_user[orden], y_user[orden]
        y_cruda = np.interp(x_base, x_user, y_user, left=y_user[0], right=y_user[-1])
        puntos_ctrl = list(zip(x_user, y_user))
    else:
        print(f"✏️ Generando curva desde puntos manuales ({TIPO_INTERPOLACION_PUNTOS})...")
        puntos_ordenados = sorted(PUNTOS_USUARIO, key=lambda p: p[0])
        x_user = np.array([p[0] for p in puntos_ordenados], dtype=np.float64)
        y_user = np.array([p[1] for p in puntos_ordenados], dtype=np.float64)
        if TIPO_INTERPOLACION_PUNTOS == "pchip":
            y_cruda = PchipInterpolator(x_user, y_user)(x_base)
        else:
            y_cruda = np.interp(x_base, x_user, y_user)
        puntos_ctrl = puntos_ordenados

    if x_user.min() > x_base.min() + 1e-9 or x_user.max() < x_base.max() - 1e-9:
        print(f"⚠️ La curva cubre λ ∈ [{x_user.min():.3f}, {x_user.max():.3f}] y el dominio es "
              f"[{x_base.min():.3f}, {x_base.max():.3f}]: los extremos se completan con el valor constante del borde.")

    # Diagnóstico de dominio (no modifica la curva)
    piso, techo = envolvente_inferior(x_base), envolvente_superior(x_base)
    debajo, encima = y_cruda < piso - 1e-9, y_cruda > techo + 1e-9
    for mask, texto in [(debajo, "debajo de la envolvente inferior"), (encima, "encima de la envolvente superior")]:
        if mask.any():
            print(f"⚠️ {mask.sum()} de {len(x_base)} puntos ({mask.mean():.1%}) {texto}, "
                  f"λ ∈ [{x_base[mask].min():.3f}, {x_base[mask].max():.3f}]: fuera del dominio de entrenamiento.")
    mono, conv = diagnostico_forma(y_cruda, x_base)
    print(f"   Forma de la curva: monótona = {mono} | convexa = {conv}")

    y_objetivo = y_cruda.copy()
    if APLICAR_PISO_ENVOLVENTE and debajo.any():
        y_objetivo = np.maximum(y_cruda, piso)
        print(f"ℹ️ Piso de envolvente aplicado en {debajo.sum()} puntos: las métricas se reportan también contra la curva cruda.")
    return y_objetivo, y_cruda, puntos_ctrl


# ── 4. AJUSTES: ÓPTIMO ACOTADO vs RED ─────────────────────────────────────────────────────────
def resolver_ajustes(A_base_ev, b_target, modelo, sx_l, sy_l, offset_l):
    coef_opt = lsq_acotado(A_base_ev, b_target)
    coef_crudo, coef_rn = predecir_coeficientes(modelo, sx_l, sy_l, offset_l, b_target)
    return coef_opt, A_base_ev @ coef_opt, coef_crudo[0], coef_rn[0], A_base_ev @ coef_rn[0]


# ── 5. GRÁFICOS ────────────────────────────────────────────────────────────────────────────────
def graficar_resultados(x, b_real, b_rn, b_opt, pts_ctrl, b_cruda=None):
    cota_inf_env, cota_sup_env = envolvente_inferior(x), envolvente_superior(x)
    rango_y = b_real.max() - b_real.min()
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

    axes[0].fill_between(x, cota_inf_env, cota_sup_env, color="#e5e7eb", alpha=0.6, label="Envolvente [mín, máx]")
    axes[0].plot(x, b_real, "k-", linewidth=2.5, label="Curva objetivo")
    if b_cruda is not None and not np.allclose(b_cruda, b_real):
        axes[0].plot(x, b_cruda, color="#6b7280", linewidth=1.5, linestyle="-.", label="Curva cruda (sin piso)")
    axes[0].plot(x, b_opt, "b:", linewidth=2.0, label=COL_OPT)
    axes[0].plot(x, b_rn, "r--", linewidth=2.0, label=f"Reconstrucción red {VERSION.upper()}")
    pts = [p for p in (pts_ctrl or []) if x.min() <= p[0] <= x.max()]
    if pts and len(pts) <= 60:
        px, py = zip(*pts)
        axes[0].scatter(px, py, color="black", zorder=5, s=25, label="Puntos de control")
    axes[0].set_ylim(-0.03 * max(b_real.max(), 1.0), 1.15 * max(b_real.max(), 1.0))
    axes[0].set_xlabel(r"Estiramiento $\lambda$", fontsize=11)
    axes[0].set_ylabel("Presión [kPa]", fontsize=11)
    axes[0].set_title(f"Curva objetivo vs. red {VERSION.upper()} [{X_MIN_DOMINIO} ≤ λ ≤ {X_MAX_DOMINIO}]",
                      fontsize=12, fontweight="bold")
    axes[0].legend(fontsize=9, loc="upper left")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, b_real - b_opt, "b:", linewidth=2.0, label=f"Residuo {COL_OPT}")
    axes[1].plot(x, b_real - b_rn, "r-", linewidth=1.8, label=f"Residuo red {VERSION.upper()}")
    axes[1].axhline(0, color="black", linestyle="--", alpha=0.6)
    axes[1].set_xlabel(r"Estiramiento $\lambda$", fontsize=11)
    axes[1].set_ylabel("Error residual (real − predicho) [kPa]", fontsize=11)
    axes[1].set_title("Residuos locales", fontsize=12, fontweight="bold")
    axes[1].legend(fontsize=9, loc="lower left")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(x, np.abs(b_real - b_opt) / rango_y * 100.0, "b:", linewidth=2.0, label=COL_OPT)
    axes[2].plot(x, np.abs(b_real - b_rn) / rango_y * 100.0, "r-", linewidth=1.8, label=f"Red {VERSION.upper()}")
    axes[2].axhline(META_PARETO_PCT, color="gray", linestyle=":", alpha=0.8, label=f"Meta Pareto {META_PARETO_PCT}%")
    axes[2].set_xlabel(r"Estiramiento $\lambda$", fontsize=11)
    axes[2].set_ylabel("Error local (% del rango)", fontsize=11)
    axes[2].set_title("Error local punto a punto", fontsize=12, fontweight="bold")
    axes[2].legend(fontsize=9, loc="upper left")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"evaluacion_inferencia_curva_j_{VERSION}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def _imprimir_metricas(titulo, m_opt, m_rn):
    col_rn = f"Red {VERSION.upper()}"
    print("\n" + "=" * 80)
    print(f"📊 {titulo}")
    print("=" * 80)
    print(f"Métrica      | {COL_OPT:<22} | {col_rn:<22}")
    print("-" * 80)
    for k in ["R2", "RMSE", "MAE", "MaxAE", "NRMSE", "MaxAE_rel"]:
        suf = "%" if k in ("NRMSE", "MaxAE_rel") else ""
        print(f"{k:<12} | {m_opt[k]:<22.6f}{suf} | {m_rn[k]:<22.6f}{suf}")
    if m_opt["NRMSE"] > 1e-4:   # el óptimo deja error apreciable (> 0.0001 % del rango)
        print(f"Exceso de RMSE de la red sobre el óptimo: {(m_rn['RMSE'] / m_opt['RMSE'] - 1.0) * 100.0:+.1f}%")
    else:
        print(f"El óptimo reproduce la curva exactamente (NRMSE ≈ 0): la red queda a {m_rn['RMSE']:.4f} kPa de RMSE.")


# ── 6. PIPELINE ─────────────────────────────────────────────────────────────────────────────────
def ejecutar_pipeline_evaluacion():
    x_base, A_ev, modelo, sx_l, sy_l, offset_l = cargar_archivos_sistema()
    n_bases_ev = A_ev.shape[1]
    y_target, y_cruda, pts_ctrl = generar_curva_objetivo(x_base)
    c_opt, y_opt, c_crudo, c_rn, y_rn = resolver_ajustes(A_ev, y_target, modelo, sx_l, sy_l, offset_l)

    mask_eval = (x_base >= X_MIN_DOMINIO) & (x_base <= X_MAX_DOMINIO)
    x_e, y_t, y_o, y_r = x_base[mask_eval], y_target[mask_eval], y_opt[mask_eval], y_rn[mask_eval]

    print("\n" + "=" * 80)
    print(f"🎯 REPORTE COMPARATIVO DE COEFICIENTES (N_BASES = {n_bases_ev})")
    print("=" * 80)
    print(f"Coeficiente | {COL_OPT:<22} | {'Red (proyectado)':<18} | {'Red (crudo)':<12} | {'Diferencia':<10}")
    print("-" * 80)
    for m in range(n_bases_ev):
        print(f"c{m+1:<10} | {c_opt[m]:<22.6f} | {c_rn[m]:<18.6f} | {c_crudo[m]:<12.6f} | {c_opt[m] - c_rn[m]:<10.6f}")

    _imprimir_metricas(f"MÉTRICAS EN [{X_MIN_DOMINIO}, {X_MAX_DOMINIO}] — curva objetivo",
                       metricas_curva(y_t, y_o), metricas_curva(y_t, y_r))
    if not np.allclose(y_cruda, y_target):
        y_c = y_cruda[mask_eval]
        _imprimir_metricas("MÉTRICAS CONTRA LA CURVA CRUDA (sin piso de envolvente)",
                           metricas_curva(y_c, y_o), metricas_curva(y_c, y_r))

    graficar_resultados(x_e, y_t, y_r, y_o, pts_ctrl, b_cruda=y_cruda[mask_eval])


ejecutar_pipeline_evaluacion()

## Célula 14b — Prueba de reproducción de las bases (4 bases × 4 amplitudes, sin piso)

In [ ]:
# ==============================================================================
# CÉLULA 14b: PRUEBA DE REPRODUCCIÓN DE LAS BASES (4 bases × 4 amplitudes, SIN piso)
# ==============================================================================
"""
Entrada: y = a · A_j (una sola base, amplitud a), sin piso de envolvente.
Respuesta exacta: c = a · e_j. Se compara la red con el valor exacto y con el óptimo acotado.
Las bases puras quedan mayormente fuera de la envolvente por construcción (por eso no se aplica
el piso); la familia 'pura' del dataset las cubre con ruido.
Requiere la Célula 14 (usa cargar_archivos_sistema y predecir_coeficientes).
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AMPLITUDES_TEST = [0.5, 1.0, 3.0, MAX_COEF]
RUIDO_REL_TEST_BASES = 0.0      # 0 = prueba limpia; p. ej. 0.005 = mismo ruido que la familia 'pura'
rng_bases = np.random.default_rng(SEED + 7)

x_b, A_b, modelo_b, sx_b, sy_b, off_b = cargar_archivos_sistema()
n_b = A_b.shape[1]

filas, curvas = [], []
for j in range(n_b):
    for a in AMPLITUDES_TEST:
        c_exacto = np.zeros(n_b)
        c_exacto[j] = a
        y = A_b @ c_exacto
        if RUIDO_REL_TEST_BASES > 0:
            y = np.maximum(y + rng_bases.normal(0.0, RUIDO_REL_TEST_BASES, size=y.shape) * np.abs(y), 0.0)
        c_crudo, c_rn = predecir_coeficientes(modelo_b, sx_b, sy_b, off_b, y)
        c_crudo, c_rn = c_crudo[0], c_rn[0]
        c_lsq = lsq_acotado(A_b, y)
        m_rn, m_lsq = metricas_curva(y, A_b @ c_rn), metricas_curva(y, A_b @ c_lsq)
        fila = {"base": j + 1, "amplitud": a, "R2_red": m_rn["R2"], "NRMSE_red_pct": m_rn["NRMSE"],
                "MaxAE_red_kPa": m_rn["MaxAE"], "R2_opt": m_lsq["R2"],
                "err_c_activo": c_rn[j] - a, "err_rel_c_activo_pct": (c_rn[j] - a) / a * 100.0,
                "max_c_inactivo": float(np.delete(c_rn, j).max()),
                "min_c_crudo": float(c_crudo.min())}
        for i in range(n_b):
            fila[f"c{i+1}_red"] = c_rn[i]
        filas.append(fila)
        curvas.append((j, a, y, A_b @ c_rn))

DF_BASES = pd.DataFrame(filas)
pd.set_option("display.width", 160)
print("\n" + "=" * 110)
print("🧱 REPRODUCCIÓN DE BASES PURAS (respuesta exacta: c = a·e_j)")
print("=" * 110)
print(DF_BASES[["base", "amplitud", "R2_red", "NRMSE_red_pct", "MaxAE_red_kPa", "err_c_activo",
                "err_rel_c_activo_pct", "max_c_inactivo", "min_c_crudo"]].to_string(index=False, float_format=lambda v: f"{v:.5f}"))
print("-" * 110)
print(f"Peor R² red: {DF_BASES['R2_red'].min():.6f} | peor |error relativo| en el coeficiente activo: "
      f"{DF_BASES['err_rel_c_activo_pct'].abs().max():.3f}% | máximo crosstalk: {DF_BASES['max_c_inactivo'].max():.5f}")
DF_BASES.to_csv(f"test_reproduccion_bases_{VERSION}.csv", index=False)
print(f"💾 'test_reproduccion_bases_{VERSION}.csv' guardado.")

fig, axes = plt.subplots(1, n_b, figsize=(4.6 * n_b, 4.2), squeeze=False)
for j in range(n_b):
    ax = axes[0, j]
    for (jj, a, y, y_rn) in curvas:
        if jj != j:
            continue
        ax.plot(x_b, y, "k-", lw=1.8, alpha=0.8)
        ax.plot(x_b, y_rn, "r--", lw=1.4)
    ax.plot([], [], "k-", label="Base exacta (a·A_j)")
    ax.plot([], [], "r--", label=f"Red {VERSION.upper()}")
    ax.set_title(f"Base {j+1} — amplitudes {AMPLITUDES_TEST}", fontsize=10, fontweight="bold")
    ax.set_xlabel(r"Estiramiento $\lambda_\theta$", fontsize=9)
    ax.set_ylabel("Presión [kPa]", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(f"test_reproduccion_bases_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()

## Célula 15 — Visualización de las bases funcionales

In [ ]:
# ==============================================================================
# CÉLULA 15: VISUALIZACIÓN DE LAS BASES FUNCIONALES (A_base)
# ==============================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for j in range(A_base.shape[1]):
    plt.plot(X_MESH_999, A_base[:, j], lw=2, label=rf"Base $A_{{{j+1}}}(\lambda_\theta)$")

plt.title(rf"Funciones base utilizadas en la reconstrucción ($N = {A_base.shape[1]}$ bases)",
          fontsize=12, fontweight="bold")
plt.xlabel(r"Estiramiento arterial $\lambda_\theta$", fontsize=10)
plt.ylabel("Valor de la base [kPa]", fontsize=10)
plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"bases_funcionales_{VERSION}.png", dpi=300, bbox_inches="tight")
plt.show()